#### Cella import dei percorsi


In [ ]:
import sys
sys.path.append("..")
import numpy as np
from pathlib import Path
from helpers import load_csv_data

data_path = Path('../data')
cache_path = data_path / 'cache'
cache_path.mkdir(exist_ok=True)   # crea la cartella se non esiste

### Caricamento con cache

In [ ]:
if (cache_path / 'x_train.npy').exists():
    # Già salvati: li leggo veloce
    x_train = np.load(cache_path / 'x_train.npy')
    x_test = np.load(cache_path / 'x_test.npy')
    y_train = np.load(cache_path / 'y_train.npy')
    train_ids = np.load(cache_path / 'train_ids.npy')
    test_ids = np.load(cache_path / 'test_ids.npy')
    print("Caricati dalla cache")
else:
    # Prima volta: leggo i CSV (lento, 1-2 minuti) e salvo
    x_train, x_test, y_train, train_ids, test_ids = load_csv_data(str(data_path))
    np.save(cache_path / 'x_train.npy', x_train)
    np.save(cache_path / 'x_test.npy', x_test)
    np.save(cache_path / 'y_train.npy', y_train)
    np.save(cache_path / 'train_ids.npy', train_ids)
    np.save(cache_path / 'test_ids.npy', test_ids)
    print("Letti dai CSV e salvati in cache")

### Header

In [ ]:
with open(data_path / 'x_train.csv', mode='r', encoding='utf-8') as f:
    headers = f.readline().strip().split(',')
headers = headers[1:]   # tolgo 'Id'

In [ ]:
print(x_train.shape, x_test.shape, y_train.shape)
print(len(headers))

## Iniziamo a togliere alcuni valori dalla tabella

Dato che 322 valori sono troppi e molte sono ingormazioni inutili, possiamo scartare colonne come:
Id, _STATE, FMONTH, IDATE, IMONTH, IDAY, IYEAR, STATERES

Amministrative (come è stata gestita la chiamata)
DISPCODE, SEQNO, _PSU, CTELENUM, PVTRESD1, CELLFON3, NUMHHOL2, CPDEMO1, QSTVER, QSTLANG, MSCODE

Pesi statistici del sondaggio (servono ai ricercatori per ricalibrare il campione, non descrivono la persona)
_STSTR, _STRWT, _RAWRAKE, _WT2RAKE, _DUALUSE, _DUALCOR, _LLCPWT

In [ ]:
da_togliere = []

da_togliere = [
    # Identificativi e date
    'Id', '_STATE', 'FMONTH', 'IDATE', 'IMONTH', 'IDAY', 'IYEAR', 'STATERES',
    
    # Amministrative (gestione della chiamata)
    'DISPCODE', 'SEQNO', '_PSU', 'CTELENUM', 'PVTRESD1', 'CELLFON3', 
    'NUMHHOL2', 'CPDEMO1', 'QSTVER', 'QSTLANG', 'MSCODE',
    
    # Pesi statistici del sondaggio
    '_STSTR', '_STRWT', '_RAWRAKE', '_WT2RAKE', '_DUALUSE', '_DUALCOR', '_LLCPWT'
]

# lista degli header da tenere
# scorre la lista headers, e ottiene sia l'inidice i che il nome, tiene solo gli indici in cui il nome non è nella lista da togliere 
keep = [i for i, name in enumerate(headers) if name not in da_togliere]

# selezioniamo le colonne, ovvero prendiamo tutte le righe " : ", ma solo con le colonne i cui indici sono specificati nella lista keep creata al primo passaggio.
x_train_clean = x_train[:, keep]

#Crea una nuova lista di nomi (names) che corrisponde esattamente alle colonne rimaste in x.
names = [headers[i] for i in keep]

# Applichiamo a x_test: 
x_test_clean = x_test[:, keep]

print(x_train_clean.shape, x_test_clean.shape, len(names))

In [ ]:
import sys
from pathlib import Path
import numpy as np

ROOT = Path.cwd().parent            # notebooks/ -> ml-project1/
sys.path.insert(0, str(ROOT))       # per importare helpers.py e cleaning.py

from helpers import load_csv_data
from cleaning import (read_header, fill_structural_zeros, recode_none_to_zero,
                      find_special_codes, special_codes_to_nan)

data_path = ROOT / "Data"           # cartella con x_train.csv, y_train.csv, x_test.csv
cache_path = ROOT / "cache"         # cartella per i file .npy
cache_path.mkdir(exist_ok=True)

if (cache_path / 'x_train.npy').exists():
    # Già salvati: li leggo veloce
    x_train = np.load(cache_path / 'x_train.npy')
    x_test = np.load(cache_path / 'x_test.npy')
    y_train = np.load(cache_path / 'y_train.npy')
    train_ids = np.load(cache_path / 'train_ids.npy')
    test_ids = np.load(cache_path / 'test_ids.npy')
    print("Caricati dalla cache")
else:
    # Prima volta: leggo i CSV (lento, 1-2 minuti) e salvo
    x_train, x_test, y_train, train_ids, test_ids = load_csv_data(str(data_path))
    np.save(cache_path / 'x_train.npy', x_train)
    np.save(cache_path / 'x_test.npy', x_test)
    np.save(cache_path / 'y_train.npy', y_train)
    np.save(cache_path / 'train_ids.npy', train_ids)
    np.save(cache_path / 'test_ids.npy', test_ids)
    print("Letti dai CSV e salvati in cache")

# nomi delle colonne: load_csv_data toglie già la colonna 'Id'
cols = read_header(data_path / "x_train.csv")
if len(cols) == x_train.shape[1] + 1:
    cols = cols[1:]
assert len(cols) == x_train.shape[1]

print("train:", x_train.shape, " test:", x_test.shape)
print(f"NaN iniziali: {np.isnan(x_train).mean():.1%}")

In [ ]:
# =====================================================================
# Arrivare al punto di Giovanni (notebooks/Giovanni.ipynb)
# Si parte dai dati GREZZI caricati nella cella precedente.
# Le celle vanno eseguite in ordine: se devi rifarle, rilancia prima la cella di caricamento.
# =====================================================================
assert x_train.shape[1] == 321, "servono i dati grezzi: rilancia prima la cella di caricamento"

def drop_columns(X, cols, names):
    """Restituisce (X senza le colonne 'names', lista dei nomi rimasti)."""
    names = set(names)
    keep = [j for j, c in enumerate(cols) if c not in names]
    return X[:, keep], [cols[j] for j in keep]

drop_a_priori = [
    # identificativi e date
    'Id', '_STATE', 'FMONTH', 'IDATE', 'IMONTH', 'IDAY', 'IYEAR', 'STATERES',
    # amministrative (gestione della chiamata)
    'DISPCODE', 'SEQNO', '_PSU', 'CTELENUM', 'PVTRESD1', 'CELLFON3',
    'NUMHHOL2', 'CPDEMO1', 'QSTVER', 'QSTLANG', 'MSCODE',
    # pesi statistici del sondaggio
    '_STSTR', '_STRWT', '_RAWRAKE', '_WT2RAKE', '_DUALUSE', '_DUALCOR', '_LLCPWT',
    # versione fisso / cellulare del questionario (il NaN dice solo il tipo di telefono)
    'COLGHOUS', 'LADULT', 'NUMADULT', 'NUMMEN', 'NUMWOMEN', 'NUMPHON2',
    'CTELNUM1', 'CELLFON2', 'CADULT', 'PVTRESD2', 'CCLGHOUS', 'CSTATE', 'LANDLINE', 'HHADULT',
    # leakage: compilata solo da chi ha avuto un infarto
    'HAREHAB1',
    # grezze ridondanti (esiste la versione pulita del CDC) e date
    'WEIGHT2', 'HEIGHT3', 'EXEROFT1', 'EXERHMM1', 'EXEROFT2', 'EXERHMM2', 'FLSHTMY2', 'HIVTSTD3',
]

x_train, cols_tr = drop_columns(x_train, cols, drop_a_priori)
x_test,  cols_te = drop_columns(x_test,  cols, drop_a_priori)
assert cols_tr == cols_te
cols = cols_tr
print("colonne rimaste:", len(cols), "(atteso 273)")

In [ ]:
# L'ordine conta: i primi due passi leggono i codici originali, il terzo li cancella
x_train = fill_structural_zeros(x_train, cols)
x_test  = fill_structural_zeros(x_test,  cols)

x_train = recode_none_to_zero(x_train, cols)
x_test  = recode_none_to_zero(x_test,  cols)

codes = find_special_codes(x_train, cols)          # codici individuati SOLO sul train...
x_train = special_codes_to_nan(x_train, cols, codes)
x_test  = special_codes_to_nan(x_test,  cols, codes)  # ...e applicati identici al test

print("NaN train: %.1f%%   test: %.1f%%   (atteso 45.6%%)" %
      (100 * np.isnan(x_train).mean(), 100 * np.isnan(x_test).mean()))

In [ ]:
SOGLIA_NAN = 0.6
CANDIDATI  = ["INSULIN", "CVDASPRN", "DRADVISE"]    # tenute da parte per la CV

nan_col = np.isnan(x_train).mean(axis=0)
drop_nan = [c for c, f in zip(cols, nan_col) if f > SOGLIA_NAN]
drop_finale = [c for c in drop_nan if c not in CANDIDATI]
print("colonne con > 60%% NaN: %d (atteso 123)  -> tolte: %d (atteso 120)" % (len(drop_nan), len(drop_finale)))

def categorizza_insulin(dia, ins):
    """0 non diabetico, 1 diabetico senza modulo, 2 ignoto, 3 diabetico + insulina, 4 diabetico senza insulina."""
    cat = np.zeros(len(dia), dtype=int)
    cat[np.isnan(dia)] = 2
    cat[dia == 1] = 1
    cat[(dia == 1) & (ins == 1)] = 3
    cat[(dia == 1) & (ins == 2)] = 4
    return cat

# INSULIN_CAT va costruita PRIMA di togliere INSULIN
ix = {c: i for i, c in enumerate(cols)}
ins_cat_tr = categorizza_insulin(x_train[:, ix["DIABETE3"]], x_train[:, ix["INSULIN"]])
ins_cat_te = categorizza_insulin(x_test[:,  ix["DIABETE3"]], x_test[:,  ix["INSULIN"]])

x_train, cols_tr = drop_columns(x_train, cols, drop_finale + ["INSULIN"])
x_test,  cols_te = drop_columns(x_test,  cols, drop_finale + ["INSULIN"])
assert cols_tr == cols_te
x_train = np.column_stack([x_train, ins_cat_tr])
x_test  = np.column_stack([x_test,  ins_cat_te])
cols = cols_tr + ["INSULIN_CAT"]
print("colonne rimaste:", len(cols), "(atteso 153)")

In [ ]:
# Colonne che sono funzione deterministica di un'altra che resta: toglierle non perde informazione
DA_TOGLIERE_COPIE = ['_RFHLTH', '_RFHYPE5', '_RFCHOL', '_LTASTH1', '_CASTHM1', '_DRDXAR1',
                     '_EDUCAG', '_INCOMG', '_RFSMOK3', 'SMOKDAY2', '_RFSEAT2', '_RFSEAT3',
                     '_AIDTST3', '_RACEG21', '_RACEGR3', '_RACE_G1', '_AGE_G', '_TOTINDA',
                     '_PA30021', '_PAINDX1', '_PASTAE1', 'HTIN4', '_RFBMI5', 'FC60_']

# Quasi costanti: NON ancora tolte, si decide in CV
DA_TOGLIERE_VUOTE = ['_FRT16', '_VEG23', '_FRUITEX', '_VEGETEX', '_FRTRESP', '_VEGRESP',
                     '_MISFRTN', '_MISVEGN', 'USENOW3', 'PAMISS1_', '_HCVU651']

x_train, cols_tr = drop_columns(x_train, cols, DA_TOGLIERE_COPIE)
x_test,  cols_te = drop_columns(x_test,  cols, DA_TOGLIERE_COPIE)
assert cols_tr == cols_te
cols = cols_tr

# Matrice BASE: senza le due colonne sull'aspirina (CVDASPRN, DRADVISE) -> ablation pronta
contese = [c for c in CANDIDATI if c in cols]
x_base,      cols_base = drop_columns(x_train, cols, contese)
x_base_test, _         = drop_columns(x_test,  cols, contese)

# --- controllo finale: devono uscire gli stessi numeri del documento di Giovanni ---
print("x_train:", x_train.shape, " x_test:", x_test.shape, "  (atteso 129 colonne)")
print("x_base :", x_base.shape,  " x_base_test:", x_base_test.shape, "  (atteso 127 colonne)")
print("NaN residui: train %.1f%%   test %.1f%%   (atteso circa 8%%)" %
      (100 * np.isnan(x_train).mean(), 100 * np.isnan(x_test).mean()))
ok = x_train.shape[1] == 129 and x_base.shape[1] == 127
print("--> stesso punto di Giovanni" if ok else "--> ATTENZIONE: numeri diversi, controlla la versione di cleaning.py")

In [ ]:
X, nomi = x_base, cols_base
nan_col = np.isnan(X).mean(axis=0)
print("celle NaN: %.1f%%   righe con almeno un NaN: %.1f%%   colonne con NaN: %d / %d" %
      (100 * np.isnan(X).mean(), 100 * np.isnan(X).any(axis=1).mean(), (nan_col > 0).sum(), len(nomi)))
for j in np.argsort(-nan_col):
    if nan_col[j] < 0.02:
        break
    print(f"   {nomi[j]:10s} {100*nan_col[j]:5.1f}%")

## Le 129 covariate del modello

Fonte: codebook BRFSS 2015. Le colonne che iniziano con `_` (e alcune che finiscono con `_`) sono **calcolate dal CDC** a partire dalle risposte grezze.

**Legenda:**
- ⚠️ codifica mista: il numero mescola unità diverse, quindi un modello lineare lo legge male
- ⓘ NaN strutturale: domanda posta solo ad alcuni
- 🔸 quasi costante, in lista `DA_TOGLIERE_VUOTE`
- ×100 / ×10: decimali impliciti; dopo la standardizzazione non conta, ma serve per leggere i valori

### 1. Salute generale (4)
| variabile | significato | valori |
|---|---|---|
| `GENHLTH` | salute generale auto-valutata | 1 eccellente … 5 scadente |
| `PHYSHLTH` | giorni negli ultimi 30 con salute fisica non buona | 0–30 |
| `MENTHLTH` | giorni negli ultimi 30 con salute mentale non buona | 0–30 |
| `POORHLTH` | giorni in cui la cattiva salute ha impedito le attività abituali | 0–30 |

### 2. Accesso alle cure (5)
| variabile | significato | valori |
|---|---|---|
| `HLTHPLN1` | ha una copertura sanitaria | 1 sì, 2 no |
| `PERSDOC2` | ha un medico di riferimento | 1 uno, 2 più di uno, 3 nessuno |
| `MEDCOST` | ha rinunciato a cure per il costo (ultimi 12 mesi) | 1 sì, 2 no |
| `CHECKUP1` | tempo dall'ultimo check-up | 1 < 1 anno … 4 ≥ 5 anni, 8 mai |
| `_HCVU651` 🔸 ⓘ | copertura sanitaria, solo per chi ha 18–64 anni | 1 sì, 2 no; gli over 65 sono diventati NaN |

### 3. Pressione e colesterolo (6)
| variabile | significato | valori |
|---|---|---|
| `BPHIGH4` | gli hanno mai detto che ha la pressione alta | 1 sì, 2 solo in gravidanza, 3 no, 4 borderline |
| `BPMEDS` ⓘ | prende farmaci per la pressione (chiesto solo se `BPHIGH4`=1) | 1 sì, 2 no |
| `BLOODCHO` | ha mai fatto il test del colesterolo | 1 sì, 2 no |
| `CHOLCHK` ⓘ | quanto tempo fa l'ultimo test (solo se `BLOODCHO`=1) | 1 < 1 anno … 4 ≥ 5 anni |
| `TOLDHI2` ⓘ | gli hanno detto che ha il colesterolo alto (solo se `BLOODCHO`=1) | 1 sì, 2 no |
| `_CHOLCHK` | ricodifica CDC del test del colesterolo | 1 negli ultimi 5 anni, 2 non negli ultimi 5, 3 mai |

### 4. Malattie croniche diagnosticate (12)
| variabile | significato | valori |
|---|---|---|
| `CVDSTRK3` | ha avuto un ictus | 1 sì, 2 no |
| `ASTHMA3` | ha mai avuto l'asma | 1 sì, 2 no |
| `_ASTHMS1` | stato dell'asma | 1 attuale, 2 passata, 3 mai |
| `CHCSCNCR` | tumore della pelle | 1 sì, 2 no |
| `CHCOCNCR` | altri tumori | 1 sì, 2 no |
| `CHCCOPD1` | BPCO, enfisema o bronchite cronica | 1 sì, 2 no |
| `HAVARTH3` | artrite (anche reumatoide, gotta, lupus, fibromialgia) | 1 sì, 2 no |
| `ADDEPEV2` | disturbo depressivo | 1 sì, 2 no |
| `CHCKIDNY` | malattia renale | 1 sì, 2 no |
| `DIABETE3` | diabete | 1 sì, 2 solo in gravidanza, 3 no, 4 prediabete |
| `DIABAGE2` | **trasformata da noi**: anni con il diabete (`_AGE80` − età alla diagnosi; 0 per i non diabetici) | continua |
| `INSULIN_CAT` | **creata da noi** da `DIABETE3` + `INSULIN` | 0 non diab., 1 diab. senza modulo, 2 ignoto, 3 diab. + insulina, 4 diab. senza insulina |

### 5. Dati demografici e socio-economici (17)
| variabile | significato | valori |
|---|---|---|
| `SEX` | sesso | 1 M, 2 F |
| `_AGE80` | età (troncata a 80) | 18–80 |
| `_AGEG5YR` | età in 13 fasce di 5 anni | 1 = 18–24 … 13 = 80+ |
| `_AGE65YR` | età in 2 fasce | 1 = 18–64, 2 = 65+ |
| `MARITAL` | stato civile | 1 sposato, 2 divorziato, 3 vedovo, 4 separato, 5 mai sposato, 6 convivente |
| `EDUCA` | istruzione | 1 nessuna … 6 laurea |
| `RENTHOM1` | casa | 1 proprietà, 2 affitto, 3 altro |
| `VETERAN3` | veterano | 1 sì, 2 no |
| `EMPLOY1` | occupazione | 1 dipendente, 2 autonomo, 3–4 disoccupato, 5 casalinga, 6 studente, 7 pensionato, 8 inabile |
| `CHILDREN` | numero di figli minorenni in casa | 0, 1, 2, … |
| `_CHLDCNT` | figli, in categorie | 1 nessuno … 6 cinque o più |
| `INCOME2` | reddito familiare annuo | 1 < 10k $ … 8 ≥ 75k $ |
| `INTERNET` | ha usato internet negli ultimi 30 giorni | 1 sì, 2 no |
| `_RACE` | razza/etnia (8 categorie) | 1 bianco non ispanico … 8 ispanico |
| `_PRACE1` | razza preferita (per chi ne dichiara più di una) | categorie |
| `_MRACE1` | razza multipla | categorie |
| `_HISPANC` | ispanico/latino | 1 sì, 2 no |

### 6. Corpo (4)
| variabile | significato | valori |
|---|---|---|
| `WTKG3` | peso | kg ×100 (7711 = 77,11 kg) |
| `HTM4` | altezza | metri ×100 (178 = 1,78 m) |
| `_BMI5` | indice di massa corporea | BMI ×100 |
| `_BMI5CAT` | BMI in categorie | 1 sottopeso, 2 normale, 3 sovrappeso, 4 obeso |

### 7. Limitazioni e disabilità (10)
| variabile | significato | valori |
|---|---|---|
| `QLACTLM2` | limitato nelle attività per problemi fisici, mentali o emotivi | 1 sì, 2 no |
| `USEEQUIP` | usa ausili (bastone, sedia a rotelle, …) | 1 sì, 2 no |
| `BLIND` | cieco o grave difficoltà a vedere | 1 sì, 2 no |
| `DECIDE` | difficoltà a concentrarsi, ricordare, decidere | 1 sì, 2 no |
| `DIFFWALK` | difficoltà a camminare o fare le scale | 1 sì, 2 no |
| `DIFFDRES` | difficoltà a vestirsi o lavarsi | 1 sì, 2 no |
| `DIFFALON` | difficoltà a sbrigare commissioni da solo | 1 sì, 2 no |
| `_LMTACT1` | l'artrite limita le attività abituali | 1 sì, 2 no, 3 senza artrite |
| `_LMTWRK1` | l'artrite limita il lavoro | 1 sì, 2 no, 3 senza artrite |
| `_LMTSCL1` | l'artrite limita la vita sociale | 1 molto, 2 un po', 3 per niente, 4 senza artrite |

### 8. Fumo (3)
| variabile | significato | valori |
|---|---|---|
| `SMOKE100` | ha fumato almeno 100 sigarette nella vita | 1 sì, 2 no |
| `_SMOKER3` | stato di fumatore | 1 ogni giorno, 2 a volte, 3 ex, 4 mai |
| `USENOW3` 🔸 | usa tabacco da masticare o snus | 1 ogni giorno, 2 a volte, 3 mai |

### 9. Alcol (9)
| variabile | significato | valori |
|---|---|---|
| `ALCDAY5` ⚠️ | giorni con alcol | 1xx = giorni a settimana, 2xx = giorni al mese, 0 = mai |
| `AVEDRNK2` | drink medi nei giorni in cui beve | numero |
| `DRNK3GE5` | volte con binge drinking (≥ 5 drink uomini / ≥ 4 donne) negli ultimi 30 gg | numero |
| `MAXDRNKS` | massimo di drink in una sola occasione | numero |
| `DRNKANY5` | ha bevuto negli ultimi 30 giorni | 1 sì, 2 no |
| `DROCDY3_` | occasioni di bere al giorno (versione pulita di `ALCDAY5`) | ×100 |
| `_RFBING5` | binge drinker | 1 no, 2 sì |
| `_DRNKWEK` | drink a settimana | ×100 |
| `_RFDRHV5` | forte bevitore (> 14 drink/settimana uomini, > 7 donne) | 1 no, 2 sì |

### 10. Frutta e verdura (24)
| variabile | significato | valori |
|---|---|---|
| `FRUITJU1`, `FRUIT1`, `FVBEANS`, `FVGREEN`, `FVORANG`, `VEGETAB1` ⚠️ | frequenza **grezza** di succo, frutta, legumi, verdura verde, arancione, altra verdura | 1xx = al giorno, 2xx = a settimana, 3xx = al mese, 0 = mai |
| `FTJUDA1_`, `FRUTDA1_`, `BEANDAY_`, `GRENDAY_`, `ORNGDAY_`, `VEGEDA1_` | le stesse sei, **pulite dal CDC**: volte al giorno | ×100 |
| `_FRUTSUM`, `_VEGESUM` | totale frutta / verdura al giorno | ×100 |
| `_FRTLT1`, `_VEGLT1` | mangia frutta / verdura almeno una volta al giorno | 1 sì, 2 no |
| `_FRT16` 🔸, `_VEG23` 🔸 | flag di valori implausibili (> 16 / > 23 al giorno) | 0/1 |
| `_FRUITEX` 🔸, `_VEGETEX` 🔸 | esclusi dal calcolo per valori mancanti o implausibili | categorie |
| `_MISFRTN` 🔸, `_MISVEGN` 🔸 | quante risposte su frutta / verdura mancano | 0, 1, 2, … |
| `_FRTRESP` 🔸, `_VEGRESP` 🔸 | risposte valide su frutta / verdura | 0/1 |

### 11. Attività fisica (28)
| variabile | significato | valori |
|---|---|---|
| `EXERANY2` | ha fatto attività fisica negli ultimi 30 giorni | 1 sì, 2 no |
| `EXRACT11`, `EXRACT21` ⓘ | **tipo** della 1ª / 2ª attività (corsa, nuoto, …) | codici nominali, decine di categorie |
| `STRENGTH` ⚠️ | esercizi di forza | 1xx = a settimana, 2xx = al mese, 0 = mai |
| `STRFREQ_` | esercizi di forza a settimana (versione pulita) | ×1000 |
| `PAFREQ1_`, `PAFREQ2_` | frequenza settimanale della 1ª / 2ª attività | ×1000 |
| `PADUR1_`, `PADUR2_` | durata di ogni sessione | minuti |
| `PAMIN11_`, `PAMIN21_` | minuti a settimana della 1ª / 2ª attività | minuti |
| `PA1MIN_` | minuti totali di attività a settimana | minuti |
| `PAVIG11_`, `PAVIG21_`, `PA1VIGM_` | minuti di attività **vigorosa** (1ª, 2ª, totale) | minuti |
| `_MINAC11`, `_MINAC21` | minuti di attività a settimana (versioni CDC) | minuti |
| `METVL11_`, `METVL21_` | intensità in MET della 1ª / 2ª attività | ×10 |
| `ACTIN11_`, `ACTIN21_` | intensità della 1ª / 2ª attività | 0 nessuna, 1 moderata, 2 vigorosa |
| `MAXVO2_` | consumo massimo di ossigeno stimato (capacità aerobica) | ×100 |
| `_PACAT1` | livello di attività | 1 molto attivo, 2 attivo, 3 insufficiente, 4 inattivo |
| `_PA150R2` | rispetta i 150 min/settimana raccomandati | 1 ≥ 150, 2 1–149, 3 zero |
| `_PA300R2` | rispetta i 300 min/settimana | 1 ≥ 300, 2 1–299, 3 zero |
| `_PASTRNG` | esercizi di forza almeno 2 volte a settimana | 1 sì, 2 no |
| `_PAREC1` | combinazione aerobica + forza | 1 entrambe … 4 nessuna |
| `PAMISS1_` 🔸 | risposte mancanti sull'attività fisica | 1 mancanti, 2 complete |

### 12. Prevenzione e comportamenti (5)
| variabile | significato | valori |
|---|---|---|
| `FLUSHOT6` | vaccino antinfluenzale negli ultimi 12 mesi | 1 sì, 2 no |
| `IMFVPLAC` ⓘ | dove ha fatto il vaccino (solo se `FLUSHOT6`=1) | codici nominali (medico, farmacia, lavoro, …) |
| `PNEUVAC3` | ha mai fatto il vaccino antipneumococco | 1 sì, 2 no |
| `HIVTST6` | ha mai fatto il test HIV | 1 sì, 2 no |
| `SEATBELT` | uso della cintura di sicurezza | 1 sempre … 5 mai, 8 non va in auto |

### 13. Candidate "contese" (2), solo in `x_train`, non in `x_base`
| variabile | significato | valori |
|---|---|---|
| `CVDASPRN` | prende aspirina ogni giorno o quasi | 1 sì, 2 no |
| `DRADVISE` | un medico gli ha consigliato di **ridurre il sale** (modulo sodio) | 1 sì, 2 no |

## Gruppo 1a: rimozione delle colonne grezze con codifica mista

### Cosa togliamo
Otto colonne che contengono risposte **grezze** del questionario in cui lo stesso numero mescola frequenza e unità di misura:

| colonna grezza | significato | codifica | versione pulita (CDC) che resta |
|---|---|---|---|
| `ALCDAY5` | giorni in cui ha bevuto alcol | 1xx = giorni a settimana, 2xx = giorni al mese, 0 = mai | `DROCDY3_` (occasioni di bere al giorno) |
| `STRENGTH` | esercizi di forza | 1xx = a settimana, 2xx = al mese, 0 = mai | `STRFREQ_` (volte a settimana) |
| `FRUITJU1` | succo di frutta 100% | 1xx = al giorno, 2xx = a settimana, 3xx = al mese, 0 = mai | `FTJUDA1_` (volte al giorno) |
| `FRUIT1` | frutta | come sopra | `FRUTDA1_` |
| `FVBEANS` | legumi | come sopra | `BEANDAY_` |
| `FVGREEN` | verdura a foglia verde | come sopra | `GRENDAY_` |
| `FVORANG` | verdura arancione | come sopra | `ORNGDAY_` |
| `VEGETAB1` | altra verdura | come sopra | `VEGEDA1_` |

### Perché
Un modello lineare legge i numeri come quantità. In `ALCDAY5`, però, 107 ("7 giorni a settimana", cioè circa 30 al mese) risulta **minore** di 201 ("1 giorno al mese"): la colonna dà al modello un'informazione sbagliata.
Il CDC ha già convertito ogni colonna in un'unità unica, e quella versione resta nel dataset.

### Cosa controlliamo prima di togliere
Vogliamo essere sicuri che la versione pulita contenga **tutta** l'informazione di quella grezza, cioè che la pulita sia una **funzione** della grezza: a ogni valore della grezza deve corrispondere **sempre lo stesso** valore della pulita. Se è così, la grezza non aggiunge nulla e toglierla non fa perdere informazione.

Per ogni coppia misuriamo:
- **purezza**: la frazione di righe in cui la pulita assume il valore "tipico" associato al valore della grezza. 1,0000 = relazione esatta. Il NaN è contato come un valore a sé, quindi si confronta anche dove mancano i dati;
- **NaN discordanti**: righe in cui una delle due è NaN e l'altra no. Se ce ne sono, il CDC ha scartato, o ricostruito, alcune risposte.

**Criterio**: la colonna grezza si toglie solo se la purezza è almeno 0,999. Altrimenti resta nel dataset e la coppia va guardata a mano.
Il controllo usa **solo il train**; le colonne tolte si tolgono poi identiche dal test.

In [ ]:
# =====================================================================
# Gruppo 1a: controllo grezza -> pulita e rimozione delle grezze
# =====================================================================
COPPIE_GREZZA_PULITA = {
    'ALCDAY5':  'DROCDY3_',
    'STRENGTH': 'STRFREQ_',
    'FRUITJU1': 'FTJUDA1_',
    'FRUIT1':   'FRUTDA1_',
    'FVBEANS':  'BEANDAY_',
    'FVGREEN':  'GRENDAY_',
    'FVORANG':  'ORNGDAY_',
    'VEGETAB1': 'VEGEDA1_',
}
SOGLIA_PUREZZA = 0.999

def purezza(a, b):
    """Frazione di righe in cui b assume il valore più frequente associato al valore di a.
    1.0 = b è una funzione esatta di a. Il NaN è trattato come un valore a sé."""
    a = np.where(np.isnan(a), -999.0, a)
    b = np.where(np.isnan(b), -999.0, b)
    _, ia = np.unique(a, return_inverse=True)
    _, ib = np.unique(b, return_inverse=True)
    nb = ib.max() + 1
    coppie, conte = np.unique(ia.astype(np.int64) * nb + ib, return_counts=True)
    gruppo_a = coppie // nb
    migliore = np.zeros(ia.max() + 1, dtype=np.int64)
    np.maximum.at(migliore, gruppo_a, conte)        # per ogni valore di a: la coppia più frequente
    return migliore.sum() / len(a)

ix = {c: i for i, c in enumerate(cols_base)}
da_togliere_1a = []

print(f"{'grezza':10s} {'pulita':10s} {'purezza':>9s} {'NaN solo grezza':>16s} {'NaN solo pulita':>16s}  esito")
for grezza, pulita in COPPIE_GREZZA_PULITA.items():
    if grezza not in ix:
        print(f"{grezza:10s} già tolta")
        continue
    if pulita not in ix:
        print(f"{grezza:10s} ATTENZIONE: manca la versione pulita {pulita} -> la grezza resta")
        continue
    a, b = x_base[:, ix[grezza]], x_base[:, ix[pulita]]
    p = purezza(a, b)
    solo_a = int((np.isnan(a) & ~np.isnan(b)).sum())    # grezza mancante, pulita presente
    solo_b = int((~np.isnan(a) & np.isnan(b)).sum())    # grezza presente, pulita mancante
    ok = p >= SOGLIA_PUREZZA
    if ok:
        da_togliere_1a.append(grezza)
    print(f"{grezza:10s} {pulita:10s} {p:9.4f} {solo_a:16d} {solo_b:16d}  {'TOLTA' if ok else 'RESTA (da guardare a mano)'}")

# --- rimozione: stesse colonne da train e test, sia per la versione base sia per quella completa ---
x_base,      cols_b = drop_columns(x_base,      cols_base, da_togliere_1a)
x_base_test, cols_bt = drop_columns(x_base_test, cols_base, da_togliere_1a)
x_train,     cols_t = drop_columns(x_train,     cols,      da_togliere_1a)
x_test,      cols_tt = drop_columns(x_test,      cols,      da_togliere_1a)
assert cols_b == cols_bt and cols_t == cols_tt, "train e test non più allineati"
cols_base, cols = cols_b, cols_t

print(f"\ntolte {len(da_togliere_1a)} colonne: {da_togliere_1a}")
print("x_base:", x_base.shape, " x_train:", x_train.shape)

## Gruppo 1b: guardiamo i numeri prima di decidere

Tre colonne **nominali** (etichette senza ordine) con molte categorie:
- `EXRACT11`: **tipo** di attività fisica principale (corsa, camminata, nuoto, …);
- `EXRACT21`: **tipo** della seconda attività (88 = nessuna seconda attività);
- `IMFVPLAC`: **dove** ha fatto il vaccino antinfluenzale (medico, farmacia, lavoro, …).

**La domanda da farsi** non è "la colonna è legata all'infarto?", ma "**aggiunge** informazione rispetto a quello che abbiamo già?":
- per `IMFVPLAC` sappiamo già *se* la persona è vaccinata (`FLUSHOT6`): guardiamo quindi i tassi **solo tra i vaccinati**, per vedere se il *luogo* fa differenza;
- per `EXRACT11/21` sappiamo già *quanto* e *quanto intensamente* la persona si muove (`ACTIN11_`, minuti a settimana): guardiamo i tassi **solo tra chi fa attività**, e come termine di paragone la stessa tabella per `ACTIN11_` (intensità).

**Come leggere le tabelle:**
- `n` = persone nella categoria, `tasso` = % di infarti, `± IC95` = margine d'errore (intervallo di confidenza al 95%);
- una differenza tra categorie è credibile solo se è **più grande del margine d'errore** e riguarda gruppi **non minuscoli**;
- `spread` = di quanto, in media, il tasso delle categorie si discosta da quello del gruppo (in punti percentuali): più è alto, più la colonna separa.

⚠️ I tassi non sono "corretti" per le altre variabili: se chi cammina ha più infarti, può dipendere dal fatto che chi cammina è in media più anziano. Questa tabella dice se vale la pena **tenere** la colonna; se aggiunge davvero qualcosa lo dirà la CV.

I codici numerici delle categorie vanno letti sul codebook BRFSS 2015.

In [ ]:
# =====================================================================
# Gruppo 1b: tassi di infarto per categoria
# =====================================================================
ix = {c: i for i, c in enumerate(cols_base)}
y01 = (y_train == 1).astype(float)

def tabella_tassi(nome, mask=None, top=None, titolo=""):
    """Tasso di infarto per ogni valore della colonna (NaN compreso), con n e IC al 95%."""
    v, y = x_base[:, ix[nome]], y01
    if mask is not None:
        v, y = v[mask], y[mask]
    base = y.mean()
    chiavi = np.where(np.isnan(v), -1, v)                     # -1 = NaN
    valori, inv, n = np.unique(chiavi, return_inverse=True, return_counts=True)
    tasso = np.bincount(inv, weights=y) / n
    ic = 1.96 * np.sqrt(tasso * (1 - tasso) / n)
    spread = 100 * np.sum(n / n.sum() * np.abs(tasso - base))

    print(f"\n=== {nome} {titolo} ===")
    print(f"persone: {len(y)}   tasso medio: {100*base:.2f}%   categorie: {len(valori)}   spread: {spread:.2f}")
    print(f"   {'valore':>7s} {'n':>8s} {'% gruppo':>9s} {'tasso':>7s} {'± IC95':>7s}")
    ordine = np.argsort(-n)
    mostrate = ordine if top is None else ordine[:top]
    for k in mostrate:
        lab = "NaN" if valori[k] == -1 else f"{valori[k]:g}"
        print(f"   {lab:>7s} {n[k]:8d} {100*n[k]/n.sum():8.1f}% {100*tasso[k]:6.2f}% {100*ic[k]:6.2f}")
    if top is not None and len(valori) > top:
        resto = ordine[top:]
        nr = n[resto].sum()
        tr = (tasso[resto] * n[resto]).sum() / nr
        print(f"   {'altre':>7s} {nr:8d} {100*nr/n.sum():8.1f}% {100*tr:6.2f}%   ({len(resto)} categorie)")

# --- IMFVPLAC: prima su tutti, poi solo tra i vaccinati ---
tabella_tassi("IMFVPLAC", titolo="(tutti: NaN = non vaccinato o intervista interrotta)")
vaccinati = x_base[:, ix["FLUSHOT6"]] == 1
tabella_tassi("IMFVPLAC", mask=vaccinati, titolo="(solo vaccinati: il LUOGO fa differenza?)")

# --- EXRACT11 / EXRACT21: solo tra chi fa attività, prime 15 categorie ---
attivi = x_base[:, ix["EXERANY2"]] == 1
tabella_tassi("EXRACT11", mask=attivi, top=15, titolo="(solo chi fa attività)")
tabella_tassi("EXRACT21", mask=attivi, top=15, titolo="(solo chi fa attività; 88 = nessuna seconda)")

# --- termine di paragone: l'intensità dell'attività, che terremo comunque ---
tabella_tassi("ACTIN11_", mask=attivi, titolo="(paragone: intensità 0 nessuna, 1 moderata, 2 vigorosa)")

## Gruppo 1b: colonne nominali con molte categorie

Tre colonne contengono **etichette senza ordine** (luoghi, tipi di attività) con molte categorie. Prima di decidere abbiamo guardato il tasso di infarto per categoria, chiedendoci non solo se la colonna è legata all'infarto, ma se **aggiunge informazione** rispetto alle colonne che teniamo già.
I codici delle categorie sono letti sul codebook BRFSS 2015.

### `IMFVPLAC`: dove ha fatto il vaccino antinfluenzale → diventa binaria `IMFV_LAVORO`

È chiesta solo a chi si è vaccinato (`FLUSHOT6` = sì), quindi il 56,8% è NaN = "non vaccinato".
Tra i vaccinati, il tasso di infarto è simile per quasi tutti i luoghi, tranne due:

| luogo | n | tasso di infarto |
|---|---|---|
| medico, negozio/farmacia, clinica, ospedale, … | 4.000–54.000 ciascuno | 10–14% |
| **8 = sul posto di lavoro** | 20.080 | **2,9%** (± 0,2) |
| **11 = a scuola** | 1.063 | **3,0%** (± 1,0) |

La differenza è enorme e non casuale, ma ha quasi certamente una spiegazione semplice: chi si vaccina al lavoro o a scuola è **più giovane e occupato**, quindi mediamente più sano. Gran parte di questa informazione è già in `_AGE80` e `EMPLOY1`.

**Decisione**: invece di 11 colonne one-hot con tassi quasi tutti uguali, teniamo **una sola colonna binaria**:
`IMFV_LAVORO` = 1 se vaccinato al lavoro o a scuola, 0 altrimenti (anche per i non vaccinati e i NaN: l'informazione "vaccinato sì/no" resta in `FLUSHOT6`).
La CV dirà se aggiunge qualcosa rispetto a età e occupazione. Se no, la togliamo, e nel report possiamo scrivere che l'effetto era spiegato dall'età.

### `EXRACT11` / `EXRACT21`: tipo della prima / seconda attività fisica → tolte

76–77 categorie (camminata, corsa, giardinaggio, …). Tra chi fa attività:

| attività | n | tasso di infarto |
|---|---|---|
| camminata (la più comune, 55%) | 120.361 | 8,7% |
| corsa / jogging | 14.714 / 2.037 | **1,2% / 2,2%** |
| pesi / yoga | 8.134 / 2.194 | **3,5% / 3,5%** |
| giardinaggio | 12.009 | 8,5% |
| cyclette | 4.960 | **12,9%** |
| golf con il cart | 2.740 | **12,1%** |

Il quadro è coerente: le attività con pochi infarti sono quelle dei **giovani**, quelle con molti infarti quelle degli **anziani**. Il tipo di attività è soprattutto un **indicatore indiretto dell'età**, che abbiamo già.
Un caso sospetto: la **cyclette** è l'attività tipica della **riabilitazione cardiaca**, quindi parte di quel 12,9% può essere una *conseguenza* dell'infarto, non una causa (lo stesso problema "post-outcome" dell'aspirina).

**Il confronto decisivo**, sulle stesse 220.487 persone:

| colonna | categorie | spread |
|---|---|---|
| `EXRACT11` (tipo di attività) | 76 | 2,11 |
| `ACTIN11_` (intensità, che teniamo) | 4 | **2,74** |

L'intensità separa **di più** con un ventesimo delle categorie: il tipo di attività non porta abbastanza informazione nuova da giustificare 75 colonne one-hot.
Per `EXRACT21` vale lo stesso (spread 2,19). Il suo valore più informativo (88 = "nessuna seconda attività") è già contenuto nei minuti della seconda attività (`PAMIN21_` = 0).

**Decisione**: togliamo `EXRACT11` e `EXRACT21`. Non le trasformiamo in una binaria come `IMFVPLAC`, perché qualsiasi raggruppamento ("attività da giovani" contro "da anziani") ricostruirebbe solo l'età.

> Nota sull'intensità (`ACTIN11_`): chi fa attività "vigorosa" ha più infarti (11,7%) di chi la fa "moderata" (5,6%). Non è un errore: il CDC calcola l'intensità **rispetto alla capacità fisica della persona** (`MAXVO2_`), che cala con l'età. La stessa camminata è "moderata" per un trentenne e "vigorosa" per un settantenne, quindi `ACTIN11_` = 2 è in buona parte un segnale di età avanzata.

In [ ]:
# =====================================================================
# Gruppo 1b: IMFVPLAC -> binaria IMFV_LAVORO; tolte EXRACT11 / EXRACT21
# =====================================================================
LUOGHI_LAVORO_SCUOLA = [8, 11]          # 8 = posto di lavoro, 11 = scuola (codebook BRFSS 2015)

# --- controllo prima della trasformazione (solo sul train) ---
ix = {c: i for i, c in enumerate(cols_base)}
if "IMFVPLAC" in ix:
    y01 = (y_train == 1).astype(float)
    luogo = x_base[:, ix["IMFVPLAC"]]
    vacc = x_base[:, ix["FLUSHOT6"]] == 1
    lav = np.isin(luogo, LUOGHI_LAVORO_SCUOLA)
    print("tasso di infarto:")
    print("   vaccinati al lavoro / a scuola : %6d persone  %5.2f%%" % (lav.sum(), 100 * y01[lav].mean()))
    print("   vaccinati altrove              : %6d persone  %5.2f%%" % ((vacc & ~lav).sum(), 100 * y01[vacc & ~lav].mean()))
    print("   non vaccinati / NaN            : %6d persone  %5.2f%%" % ((~vacc).sum(), 100 * y01[~vacc].mean()))

def trasforma_1b(X, cols):
    """IMFVPLAC -> IMFV_LAVORO (1 = vaccinato al lavoro o a scuola, 0 altrimenti); tolte EXRACT11/21.
    Usa solo i valori della riga (nessuna statistica), quindi si applica identica a train e test."""
    if "IMFVPLAC" in cols:
        lav = np.isin(X[:, cols.index("IMFVPLAC")], LUOGHI_LAVORO_SCUOLA).astype(float)  # NaN -> 0
        X, cols = drop_columns(X, cols, ["IMFVPLAC"])
        X = np.column_stack([X, lav])
        cols = cols + ["IMFV_LAVORO"]
    return drop_columns(X, cols, ["EXRACT11", "EXRACT21"])

# --- applicazione: train e test, versione base e completa ---
x_base,      cols_b  = trasforma_1b(x_base,      cols_base)
x_base_test, cols_bt = trasforma_1b(x_base_test, cols_base)
x_train,     cols_t  = trasforma_1b(x_train,     cols)
x_test,      cols_tt = trasforma_1b(x_test,      cols)
assert cols_b == cols_bt and cols_t == cols_tt, "train e test non più allineati"
cols_base, cols = cols_b, cols_t

print("\nx_base:", x_base.shape, "  x_train:", x_train.shape)
print("IMFV_LAVORO: %.1f%% di 1 nel train" % (100 * x_base[:, cols_base.index("IMFV_LAVORO")].mean()))

## Gruppo 1c: `_HCVU651` (copertura sanitaria 18–64 anni)

### Cosa togliamo
`_HCVU651` è una variabile calcolata dal CDC: "ha una copertura sanitaria", ma **solo per chi ha 18–64 anni** (sopra i 65 anni negli USA quasi tutti hanno Medicare).
Codifica: 1 = sì, 2 = no, 9 = over 65 oppure "non so".

### Perché
1. **Il NaN mescola due cose.** La nostra pulizia ha trattato il 9 come un normale "non so" e lo ha trasformato in NaN (35,9% delle righe). Ma il 9 vuol dire soprattutto "over 65". Infatti tra i NaN il tasso di infarto è del 16,3%: è l'effetto dell'**età**, non della copertura.
2. **Separa pochissimo.** Tra chi ha risposto (18–64enni) lo spread è 0,12: avere o no l'assicurazione quasi non cambia il tasso di infarto. Era già nella lista delle quasi costanti (`DA_TOGLIERE_VUOTE`).
3. **È ridondante.** Per come la costruisce il CDC, si ottiene esattamente da due colonne che teniamo:
   - età < 65 e `HLTHPLN1` = 1 (ha copertura) → 1;
   - età < 65 e `HLTHPLN1` = 2 (non ha copertura) → 2;
   - età ≥ 65 → 9 (per noi NaN).

### Cosa controlliamo prima di togliere
Che `_HCVU651` sia davvero una **funzione** di (`HLTHPLN1`, età): a ogni combinazione delle due deve corrispondere sempre lo stesso valore di `_HCVU651`. Usiamo la stessa **purezza** del gruppo 1a (1,0000 = relazione esatta).
Per l'età proviamo due versioni:
- `_AGE65YR`: fasce calcolate sull'età **dichiarata**, con NaN se non dichiarata;
- `_AGE80` ≥ 65: età **imputata** dal CDC, sempre presente.

Non sappiamo a priori quale delle due il CDC usi per `_HCVU651`, e la differenza riguarda solo chi non ha dichiarato l'età (~1,2%).

**Criterio**: si toglie se almeno una delle due versioni dà purezza ≥ 0,999. Il controllo usa solo il train; la colonna si toglie poi identica dal test.

In [ ]:
# =====================================================================
# Gruppo 1c: controllo che _HCVU651 = f(HLTHPLN1, età) e rimozione
# =====================================================================
ix = {c: i for i, c in enumerate(cols_base)}

if "_HCVU651" not in ix:
    print("_HCVU651 già tolta")
else:
    hcv  = x_base[:, ix["_HCVU651"]]
    hlth = np.nan_to_num(x_base[:, ix["HLTHPLN1"]], nan=9)        # NaN di HLTHPLN1 = valore a sé

    # chiave = combinazione (copertura, fascia d'età) in un solo numero
    chiavi = {}
    if "_AGE65YR" in ix:
        age65 = np.nan_to_num(x_base[:, ix["_AGE65YR"]], nan=9)   # 1 = 18-64, 2 = 65+, 9 = non dichiarata
        chiavi["HLTHPLN1 + _AGE65YR"] = hlth * 10 + age65
    chiavi["HLTHPLN1 + (_AGE80 >= 65)"] = hlth * 10 + (x_base[:, ix["_AGE80"]] >= 65)

    print(f"{'determinata da':28s} {'purezza':>9s}")
    risultati = {}
    for nome, k in chiavi.items():
        risultati[nome] = purezza(k, hcv)
        print(f"{nome:28s} {risultati[nome]:9.4f}")

    if max(risultati.values()) >= SOGLIA_PUREZZA:
        da_togliere_1c = ["_HCVU651"]
        x_base,      cols_b  = drop_columns(x_base,      cols_base, da_togliere_1c)
        x_base_test, cols_bt = drop_columns(x_base_test, cols_base, da_togliere_1c)
        x_train,     cols_t  = drop_columns(x_train,     cols,      da_togliere_1c)
        x_test,      cols_tt = drop_columns(x_test,      cols,      da_togliere_1c)
        assert cols_b == cols_bt and cols_t == cols_tt, "train e test non più allineati"
        cols_base, cols = cols_b, cols_t
        # non è più "in attesa della CV": la togliamo anche dalla lista delle quasi costanti
        DA_TOGLIERE_VUOTE = [c for c in DA_TOGLIERE_VUOTE if c != "_HCVU651"]
        print("\n--> _HCVU651 TOLTA (ridondante con HLTHPLN1 + età)")
    else:
        print("\n--> _HCVU651 RESTA: la relazione non è esatta, guardiamo l'output prima di decidere")

print("x_base:", x_base.shape, "  x_train:", x_train.shape)

## Gruppo 2: NaN strutturali (domande poste solo ad alcuni)

### L'idea
Alcune domande sono poste **solo a chi ha risposto "sì" a una domanda precedente** (la *madre*). Esempio: `BPMEDS` ("prende farmaci per la pressione?") è chiesta solo a chi ha risposto sì a `BPHIGH4` ("le hanno detto che ha la pressione alta?").

| persona | `BPHIGH4` | `BPMEDS` | cosa sappiamo |
|---|---|---|---|
| A | 1 = sì | 1 = sì | |
| B | 1 = sì | 2 = no | |
| C | 3 = no | NaN | domanda non posta: **sappiamo** che non prende farmaci per la pressione |
| D | NaN | NaN | intervista interrotta: **non sappiamo** nulla |

Il NaN di C non è un dato mancante: la risposta la conosciamo.

### Perché scriviamo "no" e non una nuova categoria
- **Lasciare NaN** mescolerebbe C ("non iperteso") e D ("intervista interrotta") nella stessa categoria.
- **Creare una categoria nuova** ("3 = non iperteso") produrrebbe, dopo il one-hot, una colonna **identica riga per riga** a "`BPHIGH4` = no". Due colonne uguali: il modello non sa a quale dare il peso (infinite soluzioni) e `least_squares` fallisce, perché $X^TX$ non è invertibile.
- **Scrivere "no"** è vero ("C non prende farmaci per la pressione"), non crea colonne doppie, e l'informazione "C non è iperteso" resta nella madre `BPHIGH4`. Il modello impara da `BPHIGH4` l'effetto di *avere* la pressione alta e da `BPMEDS` l'effetto *in più* dei farmaci.

I NaN veri (come D) restano NaN e verranno trattati nel preprocessing (gruppo 3).

### Le modifiche
| # | colonna | significato | modifica |
|---|---|---|---|
| 1 | `BPMEDS` | prende farmaci per la pressione (chiesta solo se `BPHIGH4` = 1) | NaN con `BPHIGH4` ∈ {2, 3, 4} → **2 = no** |
| 2 | `TOLDHI2` | gli hanno detto che ha il colesterolo alto (chiesta solo se `BLOODCHO` = 1, cioè ha fatto il test) | NaN con `BLOODCHO` = 2 → **2 = no** (non gli è mai stato detto) |
| 3 | `CHOLCHK` | quanto tempo fa l'ultimo test del colesterolo (1 < 1 anno, 2 < 2 anni, 3 < 5 anni, 4 ≥ 5 anni; chiesta solo se `BLOODCHO` = 1) | NaN con `BLOODCHO` = 2 → **4 = 5 anni o più** (il "mai" resta in `BLOODCHO`) |
| 3 | `_CHOLCHK` | riassunto CDC: 1 test negli ultimi 5 anni, 2 non negli ultimi 5, 3 mai | **tolta**: è calcolata da `BLOODCHO` + `CHOLCHK`, e la sua categoria "3 = mai" è una copia esatta di `BLOODCHO` = 2 |
| 4 | `_AGEG5YR` | età in 13 fasce di 5 anni (1 = 18–24 … 13 = 80+), NaN per chi non l'ha dichiarata (1,2%) | NaN → **fascia ricavata da `_AGE80`** (età imputata dal CDC, sempre presente) |
| 4 | `_AGE65YR` | età in 2 fasce (18–64 / 65+) | **tolta**: è solo la somma delle fasce di `_AGEG5YR` dai 65 anni in su (copia) |

L'informazione "non ha dichiarato l'età" non si perde: `MAXVO2_` è NaN esattamente sulle stesse righe e avrà il suo indicatore di missing.

### I controlli prima di modificare
1. **La domanda è davvero condizionale?** Contiamo chi ha risposto alla figlia pur avendo la madre ≠ "sì": dev'essere (quasi) zero, altrimenti l'ipotesi è sbagliata.
2. **`CHOLCHK` ha davvero i valori 1–4?** Così il 4 corrisponde a "5 anni o più".
3. **`_CHOLCHK` è una funzione di (`BLOODCHO`, `CHOLCHK`)?** Purezza ≥ 0,999, come nei gruppi 1a e 1c.
4. **La formula delle fasce d'età riproduce `_AGEG5YR`** dove esiste? Accordo ≥ 0,999.
5. **`_AGE65YR` è una funzione di `_AGEG5YR`?** Purezza ≥ 0,999.

Ogni modifica è applicata **solo se il suo controllo passa**. I controlli usano solo il train; le modifiche usano solo i valori della riga (nessuna statistica), quindi si applicano identiche a train e test.

In [ ]:
# =====================================================================
# Gruppo 2: NaN strutturali -> "no"; tolte _CHOLCHK e _AGE65YR
# =====================================================================
if "_AGE65YR" not in cols_base:
    print("gruppo 2 già applicato: niente da fare")
else:
    ix = {c: i for i, c in enumerate(cols_base)}
    col = lambda n: x_base[:, ix[n]]
    ok = {}

    def fasce_eta(age):
        """Fasce di _AGEG5YR: 1 = 18-24, 2 = 25-29, ..., 12 = 75-79, 13 = 80+."""
        return np.where(age < 25, 1, np.minimum(13, (age - 25) // 5 + 2))

    # --- CONTROLLO 1: le figlie sono chieste solo se la madre dice "sì"? ---
    print("CONTROLLO 1 - domande condizionali")
    def controllo_condizionale(figlia, madre, si):
        f, m = col(figlia), col(madre)
        risposte = ~np.isnan(f)
        fuori = risposte & ~np.isnan(m) & ~np.isin(m, si)        # ha risposto ma la madre non è "sì"
        da_riempire = np.isnan(f) & ~np.isnan(m) & ~np.isin(m, si)
        passa = fuori.sum() <= 0.001 * risposte.sum()
        print(f"   {figlia:8s} (madre {madre}): risposte con madre != sì: {fuori.sum():5d}   "
              f"NaN da riempire: {da_riempire.sum():6d}  -> {'OK' if passa else 'NON PASSA'}")
        return passa
    ok["BPMEDS"]  = controllo_condizionale("BPMEDS",  "BPHIGH4",  [1])
    ok["TOLDHI2"] = controllo_condizionale("TOLDHI2", "BLOODCHO", [1])
    ok["CHOLCHK"] = controllo_condizionale("CHOLCHK", "BLOODCHO", [1])

    # --- CONTROLLO 2: i valori di CHOLCHK sono 1-4? ---
    valori = np.unique(col("CHOLCHK")[~np.isnan(col("CHOLCHK"))])
    print("\nCONTROLLO 2 - valori di CHOLCHK:", valori)
    if not np.array_equal(valori, [1, 2, 3, 4]):
        print("   valori inattesi: CHOLCHK non viene modificata")
        ok["CHOLCHK"] = False

    # --- CONTROLLO 3: _CHOLCHK = f(BLOODCHO, CHOLCHK)? (sui valori originali) ---
    chiave = np.nan_to_num(col("BLOODCHO"), nan=9) * 10 + np.nan_to_num(col("CHOLCHK"), nan=9)
    p = purezza(chiave, col("_CHOLCHK"))
    entrambe = ~np.isnan(col("_CHOLCHK")) & ~np.isnan(col("BLOODCHO"))
    copia = np.mean((col("_CHOLCHK")[entrambe] == 3) == (col("BLOODCHO")[entrambe] == 2))
    ok["_CHOLCHK"] = p >= SOGLIA_PUREZZA
    print(f"\nCONTROLLO 3 - _CHOLCHK da (BLOODCHO, CHOLCHK): purezza {p:.4f}   "
          f"'_CHOLCHK = 3' coincide con 'BLOODCHO = 2' sul {100*copia:.2f}% delle righe")

    # --- CONTROLLO 4: la formula delle fasce riproduce _AGEG5YR? ---
    g5 = fasce_eta(col("_AGE80"))
    presenti = ~np.isnan(col("_AGEG5YR"))
    accordo = np.mean(g5[presenti] == col("_AGEG5YR")[presenti])
    ok["_AGEG5YR"] = accordo >= SOGLIA_PUREZZA
    print(f"\nCONTROLLO 4 - formula delle fasce vs _AGEG5YR: accordo {accordo:.4f}")

    # --- CONTROLLO 5: _AGE65YR = f(_AGEG5YR)? ---
    p = purezza(col("_AGEG5YR"), col("_AGE65YR"))
    ok["_AGE65YR"] = p >= SOGLIA_PUREZZA
    print(f"CONTROLLO 5 - _AGE65YR da _AGEG5YR: purezza {p:.4f}")

    # --- applicazione (solo le modifiche il cui controllo è passato) ---
    def trasforma_2(X, cols, ok):
        X = X.copy()
        j = {c: i for i, c in enumerate(cols)}
        if ok["BPMEDS"]:
            m = np.isnan(X[:, j["BPMEDS"]]) & np.isin(X[:, j["BPHIGH4"]], [2, 3, 4])
            X[m, j["BPMEDS"]] = 2                                   # 2 = non prende farmaci
        mai_testato = X[:, j["BLOODCHO"]] == 2
        if ok["TOLDHI2"]:
            m = np.isnan(X[:, j["TOLDHI2"]]) & mai_testato
            X[m, j["TOLDHI2"]] = 2                                  # 2 = non gli è mai stato detto
        if ok["CHOLCHK"]:
            m = np.isnan(X[:, j["CHOLCHK"]]) & mai_testato
            X[m, j["CHOLCHK"]] = 4                                  # 4 = 5 anni o più / mai
        if ok["_AGEG5YR"]:
            m = np.isnan(X[:, j["_AGEG5YR"]])
            X[m, j["_AGEG5YR"]] = fasce_eta(X[m, j["_AGE80"]])      # fascia dall'età imputata
        return drop_columns(X, cols, [c for c in ["_CHOLCHK", "_AGE65YR"] if ok[c]])

    nomi = ["BPMEDS", "TOLDHI2", "CHOLCHK", "_AGEG5YR"]
    prima = {c: np.isnan(col(c)).mean() for c in nomi}

    x_base,      cols_b  = trasforma_2(x_base,      cols_base, ok)
    x_base_test, cols_bt = trasforma_2(x_base_test, cols_base, ok)
    x_train,     cols_t  = trasforma_2(x_train,     cols,      ok)
    x_test,      cols_tt = trasforma_2(x_test,      cols,      ok)
    assert cols_b == cols_bt and cols_t == cols_tt, "train e test non più allineati"
    cols_base, cols = cols_b, cols_t

    print("\nRISULTATO - NaN prima -> dopo (train):")
    for c in nomi:
        dopo = np.isnan(x_base[:, cols_base.index(c)]).mean()
        print(f"   {c:9s} {100*prima[c]:5.1f}% -> {100*dopo:5.1f}%   {'modificata' if ok[c] else 'NON modificata'}")
    for c in ["_CHOLCHK", "_AGE65YR"]:
        print(f"   {c:9s} {'tolta' if ok[c] else 'NON tolta (controllo non passato)'}")
    print("\nx_base:", x_base.shape, "  x_train:", x_train.shape)

## Gruppo 3, parte A: ci sono altre colonne categoriche "doppie"?

Abbiamo già trovato diverse colonne che sono una **funzione esatta** di un'altra (`_AGE65YR` da `_AGEG5YR`, `_CHOLCHK` da `BLOODCHO` + `CHOLCHK`, `_HCVU651` da `HLTHPLN1` + età). Dopo il one-hot, queste coppie producono colonne che si ottengono **sommando altre colonne**: è collinearità esatta, che fa fallire `least_squares` e rende i pesi arbitrari.

Qui controlliamo **tutte le coppie** di colonne categoriche (il NaN conta come un valore a sé), con la stessa purezza usata finora: *B è una funzione di A* se a ogni valore di A corrisponde (quasi) sempre lo stesso valore di B.

**Attenzione alle colonne quasi costanti.** Se B vale quasi sempre la stessa cosa (es. 99,98% di zeri), la purezza è altissima per *qualsiasi* A, senza che A spieghi nulla. Per questo chiediamo anche che A **spieghi** la variabilità di B:

$$\text{frazione spiegata} = \frac{\text{purezza} - \text{quota della categoria più frequente di B}}{1 - \text{quota della categoria più frequente di B}}$$

(0 = A non aiuta più del tirare a indovinare la risposta più comune; 1 = A determina B del tutto).

**Criterio**: segnaliamo le coppie con purezza ≥ 0,999 **e** frazione spiegata ≥ 0,99.
Questa cella **non toglie nulla**: stampa le coppie trovate e decidiamo insieme.

Limite: il controllo trova le relazioni con **una sola** colonna madre. Quelle con più madri insieme (come `_CHOLCHK`) non le vede; le colonne identiche che ne derivano verranno comunque intercettate più avanti, quando costruiremo la matrice finale.

In [ ]:
# =====================================================================
# Gruppo 3, parte A: purezza tra TUTTE le coppie di colonne categoriche
# =====================================================================
# Colonne continue (gruppo 4): tutto il resto è categorico
CONTINUE = {
    '_AGE80', 'CHILDREN', 'PHYSHLTH', 'MENTHLTH', 'POORHLTH', 'DIABAGE2',
    'WTKG3', 'HTM4', '_BMI5', 'MAXVO2_',
    'AVEDRNK2', 'DRNK3GE5', 'MAXDRNKS', 'DROCDY3_', '_DRNKWEK',
    'FTJUDA1_', 'FRUTDA1_', 'BEANDAY_', 'GRENDAY_', 'ORNGDAY_', 'VEGEDA1_', '_FRUTSUM', '_VEGESUM',
    'STRFREQ_', 'PAFREQ1_', 'PAFREQ2_', 'PADUR1_', 'PADUR2_', 'PAMIN11_', 'PAMIN21_', 'PA1MIN_',
    'PAVIG11_', 'PAVIG21_', 'PA1VIGM_', '_MINAC11', '_MINAC21', 'METVL11_', 'METVL21_',
}
cat_cols = [c for c in cols_base if c not in CONTINUE]
print("colonne categoriche:", len(cat_cols), "   continue:", len(cols_base) - len(cat_cols))

# codici interi 0..k-1 per ogni colonna (NaN = valore a sé), calcolati una volta sola
codici, ncat, quota_max = {}, {}, {}
for c in cat_cols:
    v = x_base[:, cols_base.index(c)]
    _, inv, cnt = np.unique(np.where(np.isnan(v), -999.0, v), return_inverse=True, return_counts=True)
    codici[c], ncat[c], quota_max[c] = inv.astype(np.int64), len(cnt), cnt.max() / len(v)

# sanità: una "categorica" con troppi valori è probabilmente una continua classificata male
for c in cat_cols:
    if ncat[c] > 20:
        print(f"   ATTENZIONE: {c} ha {ncat[c]} valori distinti: è davvero categorica?")

# purezza per ogni coppia ordinata (A -> B) con una tabella di contingenza (bincount: veloce)
n = x_base.shape[0]
trovate = []
for a in cat_cols:
    for b in cat_cols:
        if a == b or ncat[b] == 1:
            continue
        tab = np.bincount(codici[a] * ncat[b] + codici[b],
                          minlength=ncat[a] * ncat[b]).reshape(ncat[a], ncat[b])
        p = tab.max(axis=1).sum() / n                      # per ogni valore di A, il B più frequente
        spiegata = (p - quota_max[b]) / (1 - quota_max[b])
        if p >= SOGLIA_PUREZZA and spiegata >= 0.99:
            trovate.append((b, a, p, spiegata))

coppie = {(b, a) for b, a, _, _ in trovate}
print(f"\ncoppie trovate: {len(trovate)}   (figlia = funzione esatta della madre)")
print(f"   {'figlia':10s} {'madre':10s} {'purezza':>8s} {'spiegata':>9s}  note")
for b, a, p, s in sorted(trovate):
    nota = "RECIPROCA: le due colonne sono la stessa informazione" if (a, b) in coppie else ""
    print(f"   {b:10s} {a:10s} {p:8.4f} {s:9.4f}  {nota}")

## Gruppo 3, parte A: rimozione delle colonne doppie trovate dal controllo di purezza

Il controllo di purezza tra tutte le coppie di colonne categoriche ha trovato 16 relazioni "figlia = funzione della madre". Le decisioni:

### 1. Tolta `_PA300R2`: copia esatta di `_PACAT1`
| colonna | significato |
|---|---|
| `_PACAT1` (resta) | livello di attività: 1 molto attivo (≥ 300 min/settimana), 2 attivo (150–300), 3 insufficiente, 4 inattivo |
| `_PA300R2` (tolta) | raggiunge i 300 min/settimana: sì / in parte / no |

Purezza 1,0000: `_PA300R2` è solo un raggruppamento delle categorie di `_PACAT1`. Dopo il one-hot, ogni sua colonna è una **somma di colonne di `_PACAT1`**: collinearità esatta, che fa fallire `least_squares` e rende i pesi arbitrari. Toglierla non fa perdere informazione.

### 2. Tolte le 8 colonne di servizio su frutta e verdura
| colonna | significato |
|---|---|
| `_FRT16` | flag: frutta totale > 16 volte al giorno (valore implausibile), 70 persone |
| `_VEG23` | flag: verdura totale > 23 volte al giorno, 39 persone |
| `_FRUITEX` / `_VEGETEX` | escluso dal calcolo del totale di frutta / verdura |
| `_FRTRESP` / `_VEGRESP` | ha risposto alle domande su frutta / verdura |
| `_MISFRTN` / `_MISVEGN` | quante risposte su frutta / verdura mancano |

Tre motivi:
1. **Non descrivono la salute della persona ma la qualità dei dati**: il CDC le ha create per tenere traccia di risposte mancanti o implausibili nel calcolo dei totali.
2. **Sono copie l'una dell'altra**: 13 delle 16 relazioni trovate sono tra queste colonne, e 4 coppie sono addirittura reciproche (la stessa informazione scritta in due modi). In più `_FRTRESP` = "non ha risposto" coincide esattamente con `_FRTLT1` = NaN (32.115 persone): con il NaN come categoria, sarebbe una colonna doppia. Lo stesso vale per `_VEGRESP` e `_VEGLT1`.
3. **Separano pochissimo**: spread tra 0 e 0,26, erano nella lista delle quasi costanti (`DA_TOGLIERE_VUOTE`).

L'informazione utile ("ha risposto o no su frutta/verdura") non si perde: resta nel NaN di `_FRTLT1` / `_VEGLT1` e negli indicatori di missing delle quantità.

### 3. Tenuta `SMOKE100`: quasi copia di `_SMOKER3`, ma non esatta
| colonna | significato |
|---|---|
| `_SMOKER3` | fumatore: 1 ogni giorno, 2 a volte, 3 ex fumatore, 4 mai fumato |
| `SMOKE100` | ha fumato almeno 100 sigarette nella vita: 1 sì, 2 no |

Il CDC definisce "mai fumato" proprio come "meno di 100 sigarette", quindi le due colonne coincidono quasi sempre (purezza 0,9990). Le eccezioni sono circa 330 persone, quasi certamente con `SMOKE100` risposta e `_SMOKER3` mancante.

La teniamo perché la relazione **non è esatta**: non c'è collinearità perfetta, quindi nessun metodo fallisce, e ridge e logistica regolarizzata gestiscono bene colonne molto correlate (la penalità divide il peso tra le due).
⚠️ **Da tenere d'occhio**: con `least_squares` (senza regolarizzazione) due colonne quasi identiche possono rendere i pesi instabili. Se succede, `SMOKE100` è la prima candidata da togliere.

### Stato delle quasi costanti di Giovanni
Delle 11 colonne di `DA_TOGLIERE_VUOTE`: `_HCVU651` tolta nel gruppo 1c, le 8 di frutta e verdura tolte qui. Restano in sospeso `USENOW3` e `PAMISS1_`.

In [ ]:
# =====================================================================
# Gruppo 3, parte A: rimozione di _PA300R2 e delle 8 colonne di servizio frutta/verdura
# (SMOKE100 resta: quasi copia di _SMOKER3, ma non esatta)
# =====================================================================
DA_TOGLIERE_3A = ['_PA300R2',                                   # copia esatta di _PACAT1
                  '_FRT16', '_VEG23', '_FRUITEX', '_VEGETEX',   # servizio frutta/verdura
                  '_FRTRESP', '_VEGRESP', '_MISFRTN', '_MISVEGN']

presenti = [c for c in DA_TOGLIERE_3A if c in cols_base]
if not presenti:
    print("già tolte")
else:
    x_base,      cols_b  = drop_columns(x_base,      cols_base, DA_TOGLIERE_3A)
    x_base_test, cols_bt = drop_columns(x_base_test, cols_base, DA_TOGLIERE_3A)
    x_train,     cols_t  = drop_columns(x_train,     cols,      DA_TOGLIERE_3A)
    x_test,      cols_tt = drop_columns(x_test,      cols,      DA_TOGLIERE_3A)
    assert cols_b == cols_bt and cols_t == cols_tt, "train e test non più allineati"
    cols_base, cols = cols_b, cols_t
    print(f"tolte {len(presenti)} colonne: {presenti}")

# aggiorno la lista delle quasi costanti ancora in sospeso
DA_TOGLIERE_VUOTE = [c for c in DA_TOGLIERE_VUOTE if c in cols_base]
print("quasi costanti ancora in sospeso:", DA_TOGLIERE_VUOTE)

print("categoriche:", sum(c not in CONTINUE for c in cols_base), "  continue:", sum(c in CONTINUE for c in cols_base))
print("x_base:", x_base.shape, "  x_train:", x_train.shape)

## Gruppo 3, parte B: one-hot delle categoriche, con il NaN come categoria a sé

> ⚠️ **SCELTA PROVVISORIA, DA RIVEDERE.** Trattare il NaN come categoria a sé è un primo tentativo.
> Va confrontato in CV con l'alternativa (riempire con la risposta più frequente) e, se non convince, modificato o tolto.

### Cosa facciamo
Ogni colonna categorica diventa tante colonne 0/1, una per risposta (*one-hot*). Il **NaN** viene trattato come una **risposta in più** ("non ha risposto") e ha la sua colonna.

### Perché il NaN come categoria
Anche quando i NaN sono pochissimi, sono molto informativi. Chi risponde "non so" a *"ha avuto un ictus?"* (`CVDSTRK3`, 0,2% di NaN) ha il **31,5%** di infarti, quasi 4 volte la media (8,8%); per la BPCO (`CHCCOPD1`) il 23,5%, per i reni (`CHCKIDNY`) il 22,9%. Riempire questi NaN con la risposta più comune ("no") butterebbe via questo segnale.

### Il livello di riferimento
Per ogni variabile si toglie una colonna (altrimenti le colonne di una variabile sommano a 1, come la colonna del bias: collinearità). Togliamo la colonna della **risposta più frequente**: ogni peso si legge come "quanto questa risposta si discosta da quella tipica".

### Regola d'oro
Le categorie e il riferimento si imparano **solo sul train** (`fit_categoriche`) e si applicano identici a validation e test (`transform_categoriche`). Una risposta che compare solo nel test diventa una riga di zeri, cioè viene trattata come il riferimento.
Nella cross-validation, `fit_categoriche` va chiamata **dentro ogni fold**.

In [ ]:
# =====================================================================
# Gruppo 3, parte B: one-hot delle categoriche con NaN = categoria a sé
# ⚠️ SCELTA PROVVISORIA: da confrontare in CV con "riempire con la moda"
# =====================================================================
CODICE_NAN = -1          # il NaN diventa questo valore prima del one-hot

def fit_categoriche(X, cols):
    """Impara dal train: quali colonne sono categoriche, i loro livelli, il riferimento (il più frequente)."""
    idx = np.array([j for j, c in enumerate(cols) if c not in CONTINUE], dtype=int)
    assert not (X[:, idx] == CODICE_NAN).any(), "il valore -1 esiste già nei dati: scegliere un altro CODICE_NAN"
    Xk = np.nan_to_num(X[:, idx], nan=CODICE_NAN)
    livelli, riferimento = [], []
    for c in Xk.T:
        v, n = np.unique(c, return_counts=True)
        riferimento.append(v[np.argmax(n)])
        livelli.append(np.delete(v, np.argmax(n)))          # tutti tranne il riferimento
    lab = lambda v: "NaN" if v == CODICE_NAN else f"{v:g}"
    nomi = [f"{cols[j]}={lab(v)}" for j, lev in zip(idx, livelli) for v in lev]
    return {"idx": idx, "livelli": livelli, "riferimento": riferimento, "nomi": nomi}

def transform_categoriche(X, pc, dtype=np.float32):
    """Applica il one-hot imparato sul train a qualsiasi matrice (train, validation, test)."""
    Xk = np.nan_to_num(X[:, pc["idx"]], nan=CODICE_NAN)
    return np.hstack([(Xk[:, [j]] == lev[None, :]).astype(dtype) for j, lev in enumerate(pc["livelli"])])

# --- prova: fit su tutto il train SOLO per ispezionare (nella CV va fatto dentro ogni fold) ---
pc = fit_categoriche(x_base, cols_base)
OH_tr = transform_categoriche(x_base, pc)
OH_te = transform_categoriche(x_base_test, pc)

print("categoriche:", len(pc["idx"]), " -> colonne one-hot:", OH_tr.shape[1])
print("train:", OH_tr.shape, "  test:", OH_te.shape, "  NaN nella matrice:", np.isnan(OH_tr).any())
print("colonne '=NaN' create:", sum(n.endswith("=NaN") for n in pc["nomi"]))
rare = (OH_tr.sum(axis=0) < 30).sum()
print("colonne con meno di 30 uni nel train (categorie rarissime):", rare)

# risposte presenti nel test ma mai viste nel train (diventano 'riferimento')
nuove = 0
for j, (col_j, lev, rif) in enumerate(zip(pc["idx"], pc["livelli"], pc["riferimento"])):
    visti = np.append(lev, rif)
    t = np.nan_to_num(x_base_test[:, col_j], nan=CODICE_NAN)
    nuove += (~np.isin(t, visti)).sum()
print("celle del test con una risposta mai vista nel train:", nuove)

# riferimento scelto per alcune colonne, per controllo
for c in ["GENHLTH", "CVDSTRK3", "_SMOKER3", "INCOME2", "BPMEDS"]:
    if c in cols_base:
        k = list(pc["idx"]).index(cols_base.index(c))
        print(f"   {c:9s} riferimento = {pc['riferimento'][k]:g}   altri livelli: {[f'{v:g}' if v != -1 else 'NaN' for v in pc['livelli'][k]]}")

In [ ]:
# =====================================================================
# Controllo della matrice one-hot: colonna rarissima e colonne identiche
# =====================================================================
nomi_oh = np.array(pc["nomi"])
somme = OH_tr.sum(axis=0)

print("colonne con meno di 30 uni nel train:")
for k in np.where(somme < 30)[0]:
    print(f"   {nomi_oh[k]:20s} {int(somme[k])} persone")

# colonne identiche: un'"impronta" casuale per colonna, poi verifica esatta
r = np.random.default_rng(0).random(OH_tr.shape[0]).astype(np.float32)
impronta = r @ OH_tr                           # colonne uguali -> impronta uguale
gruppi = {}
for k, h in enumerate(impronta):
    gruppi.setdefault(round(float(h), 3), []).append(k)

print("\ngruppi di colonne identiche:")
n_doppie = 0
for ks in gruppi.values():
    if len(ks) < 2:
        continue
    base = ks[0]
    uguali = [k for k in ks[1:] if np.array_equal(OH_tr[:, k], OH_tr[:, base])]
    if uguali:
        n_doppie += len(uguali)
        print(f"   {nomi_oh[base]:20s} = " + ", ".join(nomi_oh[uguali]) + f"   ({int(somme[base])} persone)")
print("colonne che sono copie esatte di un'altra:", n_doppie)

## Gruppo 3: rimozione delle colonne doppie trovate nel one-hot

Dopo il one-hot abbiamo cercato le colonne **identiche riga per riga** (copie esatte). Due colonne uguali sono un problema: il modello non sa a quale dare il peso, e `least_squares` fallisce perché $X^TX$ non è invertibile.

### Razza: teniamo solo `_RACE`
| colonna | significato | decisione |
|---|---|---|
| `_RACE` | razza ed etnia insieme (8 categorie: 1 bianco non ispanico, 2 nero non ispanico, 3 nativo americano, 4 asiatico, 5 isole del Pacifico, 6 altro, 7 multirazziale, **8 ispanico**) | **resta** |
| `_HISPANC` | ispanico/latino (1 sì, 2 no) | tolta |
| `_PRACE1` | razza "preferita" (per chi ne dichiara più di una) | tolta |
| `_MRACE1` | razza, con una categoria per i multirazziali | tolta |

Le quattro colonne descrivono la stessa cosa:
- "ispanico" (`_HISPANC` = 1) coincide riga per riga con `_RACE` = 8;
- "`_PRACE1` = NaN" coincide riga per riga con "`_MRACE1` = NaN";
- `_PRACE1` e `_MRACE1` sono altre versioni della razza, già riassunte in `_RACE`, che in più include l'etnia.

Separano poco (spread 0,4–0,8). Togliendo `_PRACE1` sparisce anche la sua categoria rarissima (codice 8: 5 persone).

### Attività fisica: tolta `_PA150R2`
| colonna | significato | decisione |
|---|---|---|
| `_PACAT1` | livello di attività: 1 molto attivo (≥ 300 min/settimana), 2 attivo (150–299), 3 insufficiente (1–149), 4 inattivo (0) | **resta** |
| `_PA150R2` | raggiunge i 150 min/settimana: 1 sì, 2 in parte (1–149), 3 no (0) | tolta |

`_PA150R2` è un raggruppamento delle categorie di `_PACAT1`: le sue categorie 2 e 3 coincidono riga per riga con le categorie 3 e 4 di `_PACAT1`, e la 1 è l'unione delle categorie 1 e 2. È lo stesso caso di `_PA300R2`, già tolta.
Si perde solo lo 0,6% di persone con `_PA150R2` noto ma `_PACAT1` mancante: per loro resta la categoria NaN di `_PACAT1`.

### Lasciate così (da tenere d'occhio)
- **`INSULIN_CAT`** (creata da Giovanni: 0 non diabetico, 1 diabetico senza modulo, 2 ignoto, 3 diabetico + insulina, 4 diabetico senza insulina): la categoria 2 coincide con "`DIABETE3` = NaN", e le categorie 1 + 3 + 4 insieme coincidono con "`DIABETE3` = 1 (diabetico)". È **collinearità esatta** con `DIABETE3`.
- **`PAMISS1_`** (il CDC segnala risposte mancanti sull'attività fisica): "`PAMISS1_` = NaN" coincide con "`EXERANY2` = NaN".

⚠️ Con ridge e logistica regolarizzata non è un problema (la penalità rende la soluzione unica). Con **`least_squares`** invece $X^TX$ resta **singolare** e il calcolo fallisce: se servirà la regressione non regolarizzata, queste due colonne andranno sistemate (unire le categorie 0/1/2 di `INSULIN_CAT`, togliere `PAMISS1_`).

In [ ]:
# =====================================================================
# Gruppo 3: tolte le colonne doppie (razza ridotta a _RACE; _PA150R2)
# INSULIN_CAT e PAMISS1_ restano: collinearità nota, da rivedere per least_squares
# =====================================================================
DA_TOGLIERE_3B = ['_HISPANC', '_PRACE1', '_MRACE1',   # razza: resta solo _RACE
                  '_PA150R2']                          # raggruppamento di _PACAT1

presenti = [c for c in DA_TOGLIERE_3B if c in cols_base]
if not presenti:
    print("già tolte")
else:
    x_base,      cols_b  = drop_columns(x_base,      cols_base, DA_TOGLIERE_3B)
    x_base_test, cols_bt = drop_columns(x_base_test, cols_base, DA_TOGLIERE_3B)
    x_train,     cols_t  = drop_columns(x_train,     cols,      DA_TOGLIERE_3B)
    x_test,      cols_tt = drop_columns(x_test,      cols,      DA_TOGLIERE_3B)
    assert cols_b == cols_bt and cols_t == cols_tt, "train e test non più allineati"
    cols_base, cols = cols_b, cols_t
    print(f"tolte {len(presenti)} colonne: {presenti}")

print("categoriche:", sum(c not in CONTINUE for c in cols_base), "  continue:", sum(c in CONTINUE for c in cols_base))
print("x_base:", x_base.shape, "  x_train:", x_train.shape)

## Gruppo 4: colonne continue (imputazione, indicatori, standardizzazione)

Le 38 colonne continue sono **quantità** (età, giorni, kg, minuti, volte al giorno): restano numeriche, non vanno in one-hot.

### P1. NaN → mediana del train + indicatore di missing
- Ogni NaN si riempie con la **mediana** della colonna, calcolata sul train. La mediana e non la media perché molte colonne hanno tantissimi zeri e una coda lunga (chi dichiara migliaia di minuti di sport): la media verrebbe tirata dalla coda, la mediana no.
- Con la sola mediana perderemmo l'informazione "**non ha risposto**", che a volte conta molto: chi non sa dire quanti giorni è stato male (`POORHLTH`) ha il 19,5% di infarti, chi non sa da quanti anni è diabetico (`DIABAGE2`) il 22,7%, contro una media dell'8,8%. Per questo aggiungiamo un **indicatore**: una colonna che vale 1 se in origine c'era un NaN, 0 altrimenti.

### P2. Un indicatore per ogni gruppo di colonne che mancano insieme
Esempio con tre colonne: A = minuti di sport, B = durata della sessione, C = giorni di malattia.

| persona | A | B | C |
|---|---|---|---|
| 1 | 120 | 30 | 0 |
| 2 | NaN | NaN | 2 |
| 3 | 90 | 45 | NaN |

A e B mancano **sempre insieme** (intervista interrotta prima delle domande sullo sport): un indicatore per A e uno per B sarebbero **colonne identiche**. Due colonne uguali non aggiungono informazione, il modello non sa a quale dare il peso, e `least_squares` fallisce.
Per questo le colonne che mancano **esattamente nelle stesse righe** formano un **gruppo**, e ogni gruppo ha **un solo indicatore**: qui uno per {A, B} e uno per {C}.

**Train e test hanno sempre le stesse colonne**: i gruppi si decidono guardando **solo il train** e poi si creano gli stessi indicatori per qualsiasi matrice. Sul test, l'indicatore di un gruppo vale 1 se **almeno una** delle sue colonne è NaN (nel train le colonne di un gruppo mancano sempre insieme, quindi lì non cambia nulla).

### P5. Standardizzazione (z-score)
Ogni colonna diventa $(x - \text{media}) / \text{deviazione standard}$, con media e deviazione standard **del train**:
- la **discesa del gradiente** usa lo stesso passo `gamma` per tutte le colonne: se `_BMI5` vale circa 2.800 e un'altra colonna 0/1, un `gamma` che va bene per una fa divergere o paralizza l'altra;
- la **regolarizzazione** $\lambda\|w\|^2$ penalizza la dimensione dei pesi, che dipende dall'unità di misura: senza standardizzare, colpirebbe le variabili in base alle unità e non alla loro importanza.

(Per `least_squares` in forma chiusa la standardizzazione non cambia le predizioni; serve per gli altri cinque metodi.)

### Regola d'oro
Mediane, gruppi, medie e deviazioni standard si imparano **solo sul train** (`fit_continue`) e si applicano identici a validation e test (`transform_continue`). Nella cross-validation, `fit_continue` va chiamata **dentro ogni fold**.

*Da completare dopo il controllo:* eventuale logaritmo e taglio dei valori estremi (P3), eventuale rimozione delle colonne che sono somme di altre (P4).

In [ ]:
# =====================================================================
# Gruppo 4: P1 (mediana + indicatore), P2 (un indicatore per gruppo), P5 (z-score)
# =====================================================================
def fit_continue(X, cols):
    """Impara dal train: mediane, gruppi di colonne che mancano insieme, media e deviazione standard."""
    idx = np.array([j for j, c in enumerate(cols) if c in CONTINUE], dtype=int)
    Xc = X[:, idx].astype(np.float64)
    mancanti = np.isnan(Xc)

    # P2: colonne con NaN raggruppate per pattern identico (stesse righe mancanti)
    con_nan = np.where(mancanti.any(axis=0))[0]
    gruppi_pos = []                      # per ogni gruppo: posizioni delle sue colonne dentro idx
    if len(con_nan) > 0:
        _, primo, gruppo = np.unique(mancanti[:, con_nan], axis=1,
                                     return_index=True, return_inverse=True)
        gruppo = np.ravel(gruppo)
        for g in np.argsort(primo):      # gruppi in ordine di prima colonna
            gruppi_pos.append(con_nan[gruppo == g])
    membri = [[cols[idx[p]] for p in pos] for pos in gruppi_pos]

    # P1: mediana del train al posto dei NaN
    med = np.nanmedian(Xc, axis=0)
    Xc = np.where(mancanti, med, Xc)

    # P5: media e deviazione standard (calcolate dopo l'imputazione)
    mu, sd = Xc.mean(axis=0), Xc.std(axis=0)
    sd[sd == 0] = 1

    nomi = ([cols[j] for j in idx]
            + [f"NaN[{m[0]}" + (f" +{len(m)-1}]" if len(m) > 1 else "]") for m in membri])
    return {"idx": idx, "gruppi_pos": gruppi_pos, "membri": membri,
            "med": med, "mu": mu, "sd": sd, "nomi": nomi}

def transform_continue(X, pk, dtype=np.float32):
    """Applica a train / validation / test le statistiche imparate sul train.
    Indicatore di un gruppo = 1 se ALMENO UNA colonna del gruppo è NaN."""
    Xc = X[:, pk["idx"]].astype(np.float64)
    miss = [np.isnan(Xc[:, pos]).any(axis=1) for pos in pk["gruppi_pos"]]
    Xc = np.where(np.isnan(Xc), pk["med"], Xc)
    Xc = (Xc - pk["mu"]) / pk["sd"]
    return np.hstack([Xc] + [m[:, None] for m in miss]).astype(dtype)

# --- prova: fit su tutto il train SOLO per ispezionare (nella CV va fatto dentro ogni fold) ---
pk = fit_continue(x_base, cols_base)
CT_tr = transform_continue(x_base, pk)
CT_te = transform_continue(x_base_test, pk)

n_cont = len(pk["idx"])
print("continue:", n_cont, " + indicatori:", len(pk["gruppi_pos"]), " -> colonne:", CT_tr.shape[1])
print("train:", CT_tr.shape, "  test:", CT_te.shape, "  NaN nella matrice:",
      np.isnan(CT_tr).any() or np.isnan(CT_te).any())
print("controllo z-score sul train: medie in [%.3f, %.3f], dev. std in [%.3f, %.3f]" % (
    CT_tr[:, :n_cont].mean(0).min(), CT_tr[:, :n_cont].mean(0).max(),
    CT_tr[:, :n_cont].std(0).min(), CT_tr[:, :n_cont].std(0).max()))

print("\ngruppi di colonne che mancano insieme (un indicatore ciascuno):")
Xte = x_base_test[:, pk["idx"]]
for k, (m, pos) in enumerate(zip(pk["membri"], pk["gruppi_pos"])):
    quota_tr = CT_tr[:, n_cont + k].mean()
    parziali_te = int((np.isnan(Xte[:, pos]).any(axis=1) & ~np.isnan(Xte[:, pos]).all(axis=1)).sum())
    print(f"   indicatore {k+1:2d}: train {100*quota_tr:5.1f}% NaN   "
          f"test, righe con NaN solo in parte del gruppo: {parziali_te:4d}   "
          f"{len(m):2d} colonne: {', '.join(m)}")

In [ ]:
# =====================================================================
# Gruppo 4: quanto si somigliano i NaN delle continue? (somiglianza di Jaccard)
# =====================================================================
nomi_c = [cols_base[j] for j in pk["idx"]]
M = np.isnan(x_base[:, pk["idx"]])
con = np.where(M.any(axis=0))[0]
Mb = M[:, con].astype(np.float32)

inter = Mb.T @ Mb                                   # persone con NaN in entrambe
cnt = Mb.sum(axis=0)                                # persone con NaN in ciascuna
J = inter / (cnt[:, None] + cnt[None, :] - inter)   # Jaccard
np.fill_diagonal(J, 0)

print("per ogni colonna: le 3 colonne con NaN più simili")
for i, k in enumerate(con):
    vicine = np.argsort(-J[i])[:3]
    testo = ", ".join(f"{nomi_c[con[v]]} {J[i, v]:.2f}" for v in vicine)
    print(f"   {nomi_c[k]:9s} {100*M[:, k].mean():5.1f}% NaN  ->  {testo}")

def gruppi_con_soglia(J, soglia):
    """Gruppi = colonne collegate da somiglianza >= soglia (anche passando per altre colonne)."""
    padre = list(range(len(J)))
    def radice(a):
        while padre[a] != a:
            a = padre[a]
        return a
    for a in range(len(J)):
        for b in range(a + 1, len(J)):
            if J[a, b] >= soglia:
                padre[radice(a)] = radice(b)
    gruppi = {}
    for a in range(len(J)):
        gruppi.setdefault(radice(a), []).append(nomi_c[con[a]])
    return list(gruppi.values())

for soglia in [0.95, 0.90, 0.80]:
    g = gruppi_con_soglia(J, soglia)
    print(f"\nsoglia {soglia}: {len(g)} indicatori")
    for m in sorted(g, key=len, reverse=True):
        if len(m) > 1:
            print(f"   {len(m):2d} colonne: {', '.join(m)}")
    print(f"   + {sum(len(m) == 1 for m in g)} colonne da sole")

## Gruppo 4: controllo delle code lunghe (P3) e delle ridondanze (P4)

Questa cella **guarda soltanto, non modifica nulla**.

**P3: valori estremi.** Per ogni continua mostriamo la quota di zeri, la mediana, i percentili 99 e 99,9, il massimo e l'**asimmetria** (*skewness*: 0 = simmetrica, valori grandi = coda lunga a destra). Una colonna con asimmetria alta e massimo molto più grande del 99,9° percentile ha valori estremi che "tirano" i pesi di un modello lineare: è candidata al logaritmo `log(1 + x)` e/o al taglio degli estremi.

**P4: somme esatte e colonne quasi uguali.** Alcune colonne dovrebbero essere somme di altre (es. minuti totali = minuti della 1ª + della 2ª attività). Una somma esatta è collinearità esatta: `least_squares` fallisce e i pesi diventano arbitrari. Verifichiamo:
- se le somme attese sono davvero esatte (sulle righe in cui tutte le colonne sono presenti);
- quali coppie di continue sono quasi identiche (correlazione > 0,99).

In [ ]:
# =====================================================================
# Gruppo 4: controllo P3 (code lunghe) e P4 (somme e colonne quasi uguali)
# =====================================================================
cont_cols = [c for c in cols_base if c in CONTINUE]
ixc = {c: cols_base.index(c) for c in cont_cols}

# --- P3: distribuzioni ---
print("P3 - distribuzioni (sui valori presenti)")
print(f"   {'colonna':10s} {'%zeri':>6s} {'mediana':>9s} {'p99':>9s} {'p99.9':>9s} {'max':>10s} {'asimm.':>7s}")
for c in cont_cols:
    v = x_base[:, ixc[c]]
    v = v[~np.isnan(v)]
    p50, p99, p999 = np.percentile(v, [50, 99, 99.9])
    s = v.std()
    asimm = np.mean(((v - v.mean()) / s) ** 3) if s > 0 else 0.0
    print(f"   {c:10s} {100*np.mean(v == 0):5.1f}% {p50:9.1f} {p99:9.1f} {p999:9.1f} {v.max():10.1f} {asimm:7.1f}")

# --- P4a: somme attese ---
SOMME = {
    "_FRUTSUM": ["FTJUDA1_", "FRUTDA1_"],
    "_VEGESUM": ["BEANDAY_", "GRENDAY_", "ORNGDAY_", "VEGEDA1_"],
    "PA1MIN_":  ["PAMIN11_", "PAMIN21_"],
    "PA1VIGM_": ["PAVIG11_", "PAVIG21_"],
}
print("\nP4a - le somme attese sono esatte? (tolleranza: 1 unità, per gli arrotondamenti)")
for tot, parti in SOMME.items():
    if tot not in ixc or any(p not in ixc for p in parti):
        print(f"   {tot}: colonne mancanti, salto")
        continue
    T = x_base[:, ixc[tot]]
    P = x_base[:, [ixc[p] for p in parti]]
    tutte = ~np.isnan(T) & ~np.isnan(P).any(axis=1)
    diff = np.abs(T[tutte] - P[tutte].sum(axis=1))
    print(f"   {tot:9s} = {' + '.join(parti):40s}  esatta sul {100*np.mean(diff <= 1):6.2f}% delle righe"
          f"   (diff. max {diff.max():.0f})")

# --- P4b: coppie quasi uguali ---
print("\nP4b - coppie di continue con correlazione > 0,99 (sulle righe in cui entrambe sono presenti)")
trovate = 0
for i, a in enumerate(cont_cols):
    for b in cont_cols[i + 1:]:
        va, vb = x_base[:, ixc[a]], x_base[:, ixc[b]]
        ok = ~np.isnan(va) & ~np.isnan(vb)
        if va[ok].std() == 0 or vb[ok].std() == 0:
            continue
        r = np.corrcoef(va[ok], vb[ok])[0, 1]
        if r > 0.99:
            trovate += 1
            print(f"   {a:10s} ~ {b:10s}  corr {r:.4f}   identiche sul {100*np.mean(va[ok] == vb[ok]):6.2f}% delle righe")
print("   coppie trovate:", trovate)

## Gruppo 4: valori fisicamente impossibili (proposta di limiti)

### L'idea
Alcune colonne contengono valori **impossibili**: 54.000 minuti di attività a settimana (una settimana ha 10.080 minuti), 52 figli minorenni in casa, 99 porzioni di frutta al giorno. Non sono persone "estreme" ma **errori** (di compilazione, di trascrizione o codici rimasti nei dati): il valore vero non lo conosciamo.
Li trasformiamo quindi in **NaN**: verranno trattati come gli altri dati mancanti (mediana + indicatore di missing).

Diverso è il caso della **coda lunga di valori veri** (es. 2.000 minuti di attività a settimana: tanti, ma possibili): quelli restano, e li comprimeremo con il logaritmo.

### I limiti proposti
| colonna | significato | limite: oltre è impossibile o sospetto |
|---|---|---|
| `CHILDREN` | figli minorenni in casa | > 15 |
| `AVEDRNK2` | drink medi nei giorni in cui beve | > 30 |
| `DRNK3GE5` | volte con binge drinking in 30 giorni | > 30 (più di una al giorno) |
| `MAXDRNKS` | massimo di drink in una sola occasione | > 40 |
| `_DRNKWEK` | drink a settimana (× 100) | > 14.000 (più di 20 drink al giorno) |
| `FTJUDA1_`, `FRUTDA1_` | succo / frutta, volte al giorno | > 16 (stessa soglia usata dal CDC per il flag `_FRT16`) |
| `BEANDAY_`, `GRENDAY_`, `ORNGDAY_`, `VEGEDA1_` | verdure, volte al giorno | > 23 (soglia del CDC per `_VEG23`) |
| `PAFREQ1_`, `PAFREQ2_` | quante volte a settimana fa la 1ª / 2ª attività | > 28 (più di 4 volte al giorno) |
| `STRFREQ_` | esercizi di forza a settimana | > 28 |
| `_MINAC11`, `_MINAC21`, `PAMIN11_`, `PAMIN21_`, `PAVIG11_`, `PAVIG21_` | minuti di attività a settimana | > 10.080 (più di 24 ore al giorno) |
| `DIABAGE2` | anni con il diabete | < 0 oppure > età (impossibile) |

Le altre continue (età, altezza, peso, BMI, giorni di cattiva salute 0–30, intensità, `MAXVO2_`, durata della sessione) non hanno valori impossibili evidenti.

I limiti sono **proposte**: questa cella mostra quante persone li superano e quali valori compaiono sopra il limite, per decidere se sono giusti.

In [ ]:
# =====================================================================
# Gruppo 4: quante persone superano i limiti fisici proposti? (solo controllo)
# =====================================================================
LIMITI = {
    "CHILDREN": 15,
    "AVEDRNK2": 30, "DRNK3GE5": 30, "MAXDRNKS": 40, "_DRNKWEK": 14000,
    "FTJUDA1_": 16, "FRUTDA1_": 16,
    "BEANDAY_": 23, "GRENDAY_": 23, "ORNGDAY_": 23, "VEGEDA1_": 23,
    "PAFREQ1_": 28, "PAFREQ2_": 28, "STRFREQ_": 28,
    "_MINAC11": 10080, "_MINAC21": 10080, "PAMIN11_": 10080, "PAMIN21_": 10080,
    "PAVIG11_": 10080, "PAVIG21_": 10080,
}
ixc = {c: cols_base.index(c) for c in cols_base if c in CONTINUE}
y01 = (y_train == 1).astype(float)

print(f"{'colonna':10s} {'limite':>7s} {'oltre':>7s} {'%':>7s} {'infarti oltre':>14s}  valori più frequenti oltre il limite (valore: persone)")
for c, lim in LIMITI.items():
    if c not in ixc:
        print(f"{c:10s} non presente")
        continue
    v = x_base[:, ixc[c]]
    oltre = v > lim                                  # i NaN danno False
    n = int(oltre.sum())
    if n:
        val, cnt = np.unique(v[oltre], return_counts=True)
        top = np.argsort(-cnt)[:5]
        testo = ", ".join(f"{val[k]:g}: {cnt[k]}" for k in top)
        tasso = f"{100*y01[oltre].mean():6.1f}%"
    else:
        testo, tasso = "-", "-"
    print(f"{c:10s} {lim:7g} {n:7d} {100*n/len(v):6.3f}% {tasso:>14s}  {testo}")

# caso speciale: anni con il diabete negativi o maggiori dell'età
d, eta = x_base[:, ixc["DIABAGE2"]], x_base[:, ixc["_AGE80"]]
neg, oltre_eta = int((d < 0).sum()), int((d > eta).sum())
print(f"\nDIABAGE2: valori negativi: {neg}   maggiori dell'età: {oltre_eta}")
if neg:
    val, cnt = np.unique(d[d < 0], return_counts=True)
    print("   valori negativi più frequenti:", ", ".join(f"{val[k]:g}: {cnt[k]}" for k in np.argsort(-cnt)[:5]))

In [ ]:
# =====================================================================
# Gruppo 4 (definitivo): totali tolti, impossibili -> NaN, mediana + indicatori, log, z-score
# =====================================================================

# --- 1-2. regole fisse: si applicano una volta a train e test ---
TOTALI = ["_FRUTSUM", "_VEGESUM", "PA1MIN_", "PA1VIGM_"]
# LIMITI: definito nella cella di controllo precedente

def valori_impossibili_a_nan(X, cols):
    """Valori oltre i limiti fisici (LIMITI) -> NaN; DIABAGE2 < 0 o > età -> NaN."""
    X = X.copy()
    j = {c: i for i, c in enumerate(cols)}
    for c, lim in LIMITI.items():
        if c in j:
            m = X[:, j[c]] > lim                    # i NaN danno False
            X[m, j[c]] = np.nan
    if "DIABAGE2" in j and "_AGE80" in j:
        d, eta = X[:, j["DIABAGE2"]], X[:, j["_AGE80"]]
        X[(d < 0) | (d > eta), j["DIABAGE2"]] = np.nan
    return X

def regole_fisse_gruppo4(X, cols):
    X, cols = drop_columns(X, cols, TOTALI)
    return valori_impossibili_a_nan(X, cols), cols

nan_prima = {c: int(np.isnan(x_base[:, cols_base.index(c)]).sum()) for c in LIMITI if c in cols_base}

x_base,      cols_b  = regole_fisse_gruppo4(x_base,      cols_base)
x_base_test, cols_bt = regole_fisse_gruppo4(x_base_test, cols_base)
x_train,     cols_t  = regole_fisse_gruppo4(x_train,     cols)
x_test,      cols_tt = regole_fisse_gruppo4(x_test,      cols)
assert cols_b == cols_bt and cols_t == cols_tt, "train e test non più allineati"
cols_base, cols = cols_b, cols_t

nuovi = {c: int(np.isnan(x_base[:, cols_base.index(c)]).sum()) - n for c, n in nan_prima.items()}
print("totali tolti:", [c for c in TOTALI])
print("valori impossibili -> NaN (train):", sum(nuovi.values()), "celle in", sum(v > 0 for v in nuovi.values()), "colonne")
print("x_base:", x_base.shape, "  x_train:", x_train.shape)

# --- 3-5. preprocessing da fare dentro ogni fold ---
LOG_COLS = {
    "AVEDRNK2", "DRNK3GE5", "MAXDRNKS", "_DRNKWEK",                              # alcol
    "FTJUDA1_", "FRUTDA1_", "BEANDAY_", "GRENDAY_", "ORNGDAY_", "VEGEDA1_",      # frutta e verdura
    "PADUR1_", "PADUR2_", "PAFREQ1_", "PAFREQ2_", "STRFREQ_",                    # attività fisica
    "_MINAC11", "_MINAC21", "PAMIN11_", "PAMIN21_", "PAVIG11_", "PAVIG21_",
}
USA_LOG = True       # ⚠️ scelta da confrontare in CV (con / senza logaritmo)

def fit_continue(X, cols, usa_log=None):
    """Impara dal train: gruppi di NaN, mediane, quali colonne vanno in log, media e deviazione standard."""
    usa_log = USA_LOG if usa_log is None else usa_log
    idx = np.array([j for j, c in enumerate(cols) if c in CONTINUE], dtype=int)
    log_mask = np.array([usa_log and cols[j] in LOG_COLS for j in idx], dtype=bool)
    Xc = X[:, idx].astype(np.float64)
    mancanti = np.isnan(Xc)

    # 3a. un indicatore per ogni gruppo di colonne con NaN esattamente nelle stesse righe
    con_nan = np.where(mancanti.any(axis=0))[0]
    gruppi_pos = []
    if len(con_nan) > 0:
        _, primo, gruppo = np.unique(mancanti[:, con_nan], axis=1,
                                     return_index=True, return_inverse=True)
        gruppo = np.ravel(gruppo)
        for g in np.argsort(primo):
            gruppi_pos.append(con_nan[gruppo == g])
    membri = [[cols[idx[p]] for p in pos] for pos in gruppi_pos]

    # 3b. mediana del train al posto dei NaN
    med = np.nanmedian(Xc, axis=0)
    Xc = np.where(mancanti, med, Xc)

    # 4. logaritmo per le colonne con coda lunga
    assert (Xc[:, log_mask] >= 0).all(), "valori negativi in una colonna da mettere in log"
    Xc[:, log_mask] = np.log1p(Xc[:, log_mask])

    # 5. media e deviazione standard
    mu, sd = Xc.mean(axis=0), Xc.std(axis=0)
    sd[sd == 0] = 1

    nomi = ([("log " if l else "") + cols[j] for j, l in zip(idx, log_mask)]
            + [f"NaN[{m[0]}" + (f" +{len(m)-1}]" if len(m) > 1 else "]") for m in membri])
    return {"idx": idx, "log_mask": log_mask, "gruppi_pos": gruppi_pos, "membri": membri,
            "med": med, "mu": mu, "sd": sd, "nomi": nomi}

def transform_continue(X, pk, dtype=np.float32):
    """Applica a train / validation / test quanto imparato sul train.
    Indicatore di un gruppo = 1 se ALMENO UNA colonna del gruppo è NaN."""
    Xc = X[:, pk["idx"]].astype(np.float64)
    miss = [np.isnan(Xc[:, pos]).any(axis=1) for pos in pk["gruppi_pos"]]
    Xc = np.where(np.isnan(Xc), pk["med"], Xc)
    Xc[:, pk["log_mask"]] = np.log1p(np.maximum(Xc[:, pk["log_mask"]], 0))
    Xc = (Xc - pk["mu"]) / pk["sd"]
    return np.hstack([Xc] + [m[:, None] for m in miss]).astype(dtype)

# --- prova: fit su tutto il train SOLO per ispezionare (nella CV va fatto dentro ogni fold) ---
pk = fit_continue(x_base, cols_base)
CT_tr = transform_continue(x_base, pk)
CT_te = transform_continue(x_base_test, pk)

n_cont = len(pk["idx"])
print("\ncontinue:", n_cont, "(di cui in log:", int(pk["log_mask"].sum()), ") + indicatori:",
      len(pk["gruppi_pos"]), " -> colonne:", CT_tr.shape[1])
print("train:", CT_tr.shape, "  test:", CT_te.shape, "  NaN nella matrice:",
      np.isnan(CT_tr).any() or np.isnan(CT_te).any())

# effetto del logaritmo sull'asimmetria (train)
def asimmetria(v):
    s = v.std()
    return np.mean(((v - v.mean()) / s) ** 3) if s > 0 else 0.0
print("\nasimmetria prima -> dopo il logaritmo (train, dopo l'imputazione):")
for k in np.where(pk["log_mask"])[0]:
    grezza = np.where(np.isnan(x_base[:, pk["idx"][k]]), pk["med"][k], x_base[:, pk["idx"][k]])
    print(f"   {pk['nomi'][k]:14s} {asimmetria(grezza):6.1f} -> {asimmetria(CT_tr[:, k]):5.1f}")

## Matrice finale: categoriche + continue, e controllo delle collinearità

### Come è fatta
La matrice che diamo ai modelli è l'unione di tre blocchi:

| blocco | da dove | contenuto |
|---|---|---|
| bias | — | una colonna di 1 |
| continue | `fit_continue` / `transform_continue` | z-score (con log per le code lunghe) + indicatori di missing |
| categoriche | `fit_categoriche` / `transform_categoriche` | one-hot, NaN come categoria (⚠️ provvisorio), riferimento = risposta più frequente |

`fit_prep` impara tutto dal train, `transform_prep` lo applica a qualsiasi matrice. Nella cross-validation `fit_prep` va chiamata **dentro ogni fold**.

### Il controllo: c'è ancora collinearità esatta?
Abbiamo tolto molte colonne doppie, ma alcune dipendenze si vedono solo **combinando più colonne**: ad esempio, le categorie 1 + 3 + 4 di `INSULIN_CAT` sommate danno esattamente "`DIABETE3` = 1".
Il modo sistematico per trovarle è guardare $X^TX$:
- se è **invertibile** (rango pieno), `least_squares` ha una soluzione unica;
- se ha **autovalori (quasi) nulli**, esiste una combinazione di colonne che vale (quasi) sempre zero: è una collinearità. Il suo **autovettore** dice quali colonne sono coinvolte.

Stampiamo:
- il rango e il **numero di condizionamento** (rapporto tra il più grande e il più piccolo autovalore: oltre 10¹⁰ la matrice è in pratica singolare);
- per ogni autovalore quasi nullo, le colonne con il peso maggiore nel suo autovettore.

Ci aspettiamo di ritrovare le due collinearità note (`INSULIN_CAT`/`DIABETE3` e `PAMISS1_`/`EXERANY2`).

In [ ]:
# =====================================================================
# Matrice finale (bias + continue + categoriche) e controllo del rango di X^T X
# =====================================================================
def fit_prep(X, cols):
    """Impara dal train tutto il preprocessing (continue + categoriche)."""
    pk = fit_continue(X, cols)
    pc = fit_categoriche(X, cols)
    nomi = ["bias"] + pk["nomi"] + pc["nomi"]
    return {"pk": pk, "pc": pc, "nomi": nomi}

def transform_prep(X, p, dtype=np.float32):
    """Applica a train / validation / test il preprocessing imparato sul train."""
    return np.hstack([np.ones((X.shape[0], 1), dtype=dtype),
                      transform_continue(X, p["pk"], dtype),
                      transform_categoriche(X, p["pc"], dtype)])

# --- prova: fit su tutto il train SOLO per ispezionare ---
p = fit_prep(x_base, cols_base)
X_tr = transform_prep(x_base, p)
X_te = transform_prep(x_base_test, p)
nomi = np.array(p["nomi"])

print("matrice finale: train", X_tr.shape, "  test", X_te.shape)
print(f"   bias 1 + continue {len(p['pk']['idx'])} + indicatori {len(p['pk']['gruppi_pos'])} "
      f"+ one-hot {len(p['pc']['nomi'])}")
print("NaN:", np.isnan(X_tr).any() or np.isnan(X_te).any())
print("memoria train: %.0f MB" % (X_tr.nbytes / 1e6))

# --- X^T X accumulata a blocchi in float64 (evita di copiare tutta la matrice) ---
d = X_tr.shape[1]
G = np.zeros((d, d))
for s in range(0, X_tr.shape[0], 50_000):
    B = X_tr[s:s + 50_000].astype(np.float64)
    G += B.T @ B

# per confrontare colonne con scale diverse: si lavora sulla versione normalizzata
dg = np.sqrt(np.diag(G))
Gn = G / np.outer(dg, dg)
autoval, autovet = np.linalg.eigh(Gn)          # autovalori in ordine crescente

print(f"\nrango numerico: {np.sum(autoval > 1e-10 * autoval[-1])} su {d} colonne")
print(f"numero di condizionamento: {autoval[-1] / max(autoval[0], 1e-300):.2e}")

print("\nautovalori quasi nulli (< 1e-8) e colonne coinvolte:")
quasi_nulli = np.where(autoval < 1e-8 * autoval[-1])[0]
if len(quasi_nulli) == 0:
    print("   nessuno: X^T X è invertibile, nessuna collinearità esatta")
for k in quasi_nulli:
    v = autovet[:, k]
    top = np.argsort(-np.abs(v))[:6]
    coinvolte = ", ".join(f"{nomi[j]} ({v[j]:+.2f})" for j in top if abs(v[j]) > 0.05)
    print(f"   autovalore {autoval[k]:.1e}:  {coinvolte}")

print("\ni 10 autovalori più piccoli:", ", ".join(f"{a:.1e}" for a in autoval[:10]))

## Collinearità esatte: correzioni sulle colonne (D, E, F, G)

Il controllo del rango di $X^TX$ ha trovato 17 collinearità esatte (rango 258 su 275 colonne). Quattro gruppi si risolvono correggendo le colonne con regole fisse (nessuna statistica), quindi prima della CV e identiche su train e test.

### D. `INSULIN_CAT`: teniamo solo l'informazione sull'insulina
`INSULIN_CAT` (creata da noi: 0 non diabetico, 1 diabetico senza modulo, 2 ignoto, 3 diabetico + insulina, 4 diabetico senza insulina) ripeteva lo stato del diabete già contenuto in `DIABETE3`:
- "`INSULIN_CAT` = 2" coincide con "`DIABETE3` mancante";
- le categorie 1 + 3 + 4 insieme coincidono con "`DIABETE3` = 1 (diabetico)".

Uniamo 0, 1 e 2 in **0 = "non risulta che usi insulina"**. Restano 0 / 3 = diabetico con insulina / 4 = diabetico senza insulina. Lo stato del diabete resta in `DIABETE3`, e `INSULIN_CAT` dice solo l'effetto **in più** dell'insulina (stessa logica del gruppo 2).

### E. Tolta `SMOKE100` (ha fumato almeno 100 sigarette)
La purezza con `_SMOKER3` (fumatore: ogni giorno / a volte / ex / mai) era 0,9990, non 1. Ma nel one-hot le ~330 eccezioni (`SMOKE100` = sì, `_SMOKER3` mancante) sono compensate esattamente dalla colonna "`_SMOKER3` = NaN":
`SMOKE100=1` = `_SMOKER3=1` + `=2` + `=3` + (`_SMOKER3=NaN` − `SMOKE100=NaN`).
Tutta la sua informazione è già in `_SMOKER3`.

### F. Artrite: "non ha l'artrite" → "non lo limita"
| colonna | significato | categoria "non ha l'artrite" |
|---|---|---|
| `HAVARTH3` | ha l'artrite (1 sì, 2 no) | — |
| `_LMTACT1` | l'artrite limita le attività abituali (1 sì, 2 no) | 3 |
| `_LMTWRK1` | l'artrite limita il lavoro (1 sì, 2 no) | 3 |
| `_LMTSCL1` | l'artrite limita la vita sociale (1 molto, 2 un po', 3 per niente) | 4 |

La categoria "non ha l'artrite" è una **copia** di `HAVARTH3` = 2 (290.621 persone in entrambe, anche nel codebook): tre colonne doppie. Come per `BPMEDS` nel gruppo 2, per chi non ha l'artrite la risposta implicita è "**non lo limita**": 3 → 2 in `_LMTACT1` e `_LMTWRK1`, 4 → 3 in `_LMTSCL1`. `HAVARTH3` dice se c'è l'artrite, le `_LMT` l'effetto **in più** della limitazione.

### G. Tolta `ASTHMA3` (ha mai avuto l'asma)
`_ASTHMS1` (1 asma attuale, 2 passata, 3 mai) è costruita dal CDC proprio da `ASTHMA3`: "mai" = `ASTHMA3` = 2. `_ASTHMS1` contiene tutta l'informazione di `ASTHMA3` e in più distingue "attuale" da "passata".

### Controlli
Ogni modifica è applicata solo se la relazione che la giustifica è **esatta** sul train.

In [ ]:
# =====================================================================
# Collinearità esatte: correzioni D (INSULIN_CAT), E (SMOKE100), F (artrite), G (ASTHMA3)
# =====================================================================
if any(c in cols_base for c in ["PA1MIN_", "PA1VIGM_", "_FRUTSUM", "_VEGESUM"]):
    raise RuntimeError("Prima esegui la cella DEFINITIVA del gruppo 4 (toglie i 4 totali)")

ix = {c: i for i, c in enumerate(cols_base)}
col = lambda n: x_base[:, ix[n]]
ok = {}
uguali = lambda a, b: bool(np.array_equal(a, b))

# --- D: INSULIN_CAT ---
if "INSULIN_CAT" in ix and len(np.unique(col("INSULIN_CAT"))) == 5:
    ins, dia = col("INSULIN_CAT"), col("DIABETE3")
    c1 = uguali(ins == 2, np.isnan(dia))
    c2 = uguali(np.isin(ins, [1, 3, 4]), dia == 1)
    ok["D"] = c1 and c2
    print(f"D  INSULIN_CAT=2 <-> DIABETE3 mancante: {c1}   INSULIN_CAT in {{1,3,4}} <-> DIABETE3=1: {c2}")
else:
    ok["D"] = False
    print("D  INSULIN_CAT già ricodificata (o assente)")

# --- E: SMOKE100 ---
ok["E"] = "SMOKE100" in ix
print("E  SMOKE100:", "da togliere" if ok["E"] else "già tolta")

# --- F: artrite ---
NO_ARTRITE = {"_LMTACT1": 3, "_LMTWRK1": 3, "_LMTSCL1": 4}
ok["F"] = {}
for c, codice in NO_ARTRITE.items():
    if c in ix:
        r = uguali(col(c) == codice, col("HAVARTH3") == 2)
        ok["F"][c] = r
        print(f"F  {c}={codice} <-> HAVARTH3=2: {r}   ({int((col(c) == codice).sum())} persone)")

# --- G: ASTHMA3 ---
if "ASTHMA3" in ix:
    ok["G"] = uguali(col("_ASTHMS1") == 3, col("ASTHMA3") == 2)
    print(f"G  _ASTHMS1=3 <-> ASTHMA3=2: {ok['G']}")
else:
    ok["G"] = False
    print("G  ASTHMA3 già tolta")

# --- applicazione (solo dove il controllo è passato) ---
def correggi_collinearita(X, cols, ok):
    X = X.copy()
    j = {c: i for i, c in enumerate(cols)}
    if ok["D"]:
        m = np.isin(X[:, j["INSULIN_CAT"]], [0, 1, 2])
        X[m, j["INSULIN_CAT"]] = 0                       # 0 = non risulta che usi insulina
    for c, passa in ok["F"].items():
        if passa:
            codice = NO_ARTRITE[c]
            m = X[:, j[c]] == codice
            X[m, j[c]] = codice - 1                      # "non ha l'artrite" -> "non lo limita"
    da_togliere = (["SMOKE100"] if ok["E"] else []) + (["ASTHMA3"] if ok["G"] else [])
    return drop_columns(X, cols, da_togliere)

x_base,      cols_b  = correggi_collinearita(x_base,      cols_base, ok)
x_base_test, cols_bt = correggi_collinearita(x_base_test, cols_base, ok)
x_train,     cols_t  = correggi_collinearita(x_train,     cols,      ok)
x_test,      cols_tt = correggi_collinearita(x_test,      cols,      ok)
assert cols_b == cols_bt and cols_t == cols_tt, "train e test non più allineati"
cols_base, cols = cols_b, cols_t

print("\nINSULIN_CAT valori:", np.unique(x_base[:, cols_base.index("INSULIN_CAT")]))
for c in NO_ARTRITE:
    v = x_base[:, cols_base.index(c)]
    print(f"{c} valori:", np.unique(v[~np.isnan(v)]))
print("x_base:", x_base.shape, "  x_train:", x_train.shape)

## Matrice finale: categoriche + continue, senza colonne doppie

### Come è fatta
| blocco | contenuto |
|---|---|
| bias | una colonna di 1 |
| continue | z-score (con log per le code lunghe) + indicatori di missing |
| categoriche | one-hot, NaN come categoria (⚠️ provvisorio), riferimento = risposta più frequente |

### B + C. Eliminazione automatica delle colonne identiche
Abbiamo controllato le copie dentro ciascun blocco, ma non **tra** i blocchi. Il controllo del rango ha trovato che alcune colonne "= NaN" delle categoriche coincidono riga per riga con l'indicatore di missing di una continua dello stesso modulo del questionario (es. `DRNKANY5=NaN` "non ha detto se beve" = `NaN[DROCDY3_]`; `_BMI5CAT=NaN` = `NaN[_BMI5]`), e che `PAMISS1_=NaN` = `EXERANY2=NaN`.

Invece di cercarle a mano, `fit_prep` costruisce la matrice e:
1. trova le colonne **identiche** sul train (un'"impronta" casuale per colonna, poi verifica esatta);
2. di ogni gruppo ne **tiene una**, preferendo nell'ordine: bias, continue, categoriche, indicatori. Quindi tra `DRNKANY5=NaN` e `NaN[DROCDY3_]` resta la categorica;
3. toglie anche le colonne **costanti** (tranne il bias), che possono comparire in un fold della CV se una categoria rara non c'è.

Le colonne da tenere si decidono **sul train** e si applicano identiche a validation e test, quindi le dimensioni coincidono sempre. Nella CV, `fit_prep` va chiamata dentro ogni fold.

### Controllo del rango
Se $X^TX$ non ha autovalori (quasi) nulli, la matrice ha **rango pieno**: nessuna collinearità esatta, `least_squares` ha una soluzione unica.

In [ ]:
# =====================================================================
# Matrice finale: bias + continue + categoriche, senza colonne identiche o costanti
# =====================================================================
PRIORITA = {"bias": 0, "cont": 1, "oh": 2, "ind": 3}     # a parità di colonna, chi si tiene

def _blocchi(X, pk, pc, dtype=np.float32):
    return np.hstack([np.ones((X.shape[0], 1), dtype=dtype),
                      transform_continue(X, pk, dtype),
                      transform_categoriche(X, pc, dtype)])

def fit_prep(X, cols, verbose=False):
    """Impara dal train tutto il preprocessing e quali colonne tenere."""
    pk = fit_continue(X, cols)
    pc = fit_categoriche(X, cols)
    nomi = np.array(["bias"] + pk["nomi"] + pc["nomi"])
    tipo = np.array(["bias"] + ["cont"] * len(pk["idx"]) + ["ind"] * len(pk["gruppi_pos"])
                    + ["oh"] * len(pc["nomi"]))
    Z = _blocchi(X, pk, pc)

    # 1. colonne costanti (tranne il bias)
    costante = Z.max(axis=0) == Z.min(axis=0)
    costante[0] = False

    # 2. colonne identiche: impronta = proiezione su un vettore casuale (accumulata a blocchi)
    r = np.random.default_rng(0).standard_normal(Z.shape[0])
    impronta = np.zeros(Z.shape[1])
    for s in range(0, Z.shape[0], 50_000):
        impronta += r[s:s + 50_000] @ Z[s:s + 50_000].astype(np.float64)
    gruppi = {}
    for k in np.where(~costante)[0]:
        gruppi.setdefault(impronta[k], []).append(k)

    togli, copie = set(np.where(costante)[0]), []
    for ks in gruppi.values():
        if len(ks) < 2:
            continue
        ks = sorted(ks, key=lambda k: (PRIORITA[tipo[k]], k))
        tenuta = ks[0]
        for k in ks[1:]:
            if np.array_equal(Z[:, k], Z[:, tenuta]):
                togli.add(k)
                copie.append((nomi[k], nomi[tenuta]))

    keep = np.array([k for k in range(Z.shape[1]) if k not in togli])
    if verbose:
        print(f"colonne costanti tolte: {int(costante.sum())}", list(nomi[costante]))
        print(f"colonne identiche tolte: {len(copie)}")
        for tolta, tenuta in copie:
            print(f"   {tolta:22s} = {tenuta}  (tenuta)")
    return {"pk": pk, "pc": pc, "keep": keep, "nomi": list(nomi[keep])}

def transform_prep(X, p, dtype=np.float32):
    """Applica a train / validation / test il preprocessing imparato sul train."""
    return _blocchi(X, p["pk"], p["pc"], dtype)[:, p["keep"]]

# --- prova: fit su tutto il train SOLO per ispezionare ---
p = fit_prep(x_base, cols_base, verbose=True)
X_tr = transform_prep(x_base, p)
X_te = transform_prep(x_base_test, p)
nomi = np.array(p["nomi"])

print("\nmatrice finale: train", X_tr.shape, "  test", X_te.shape)
print("NaN:", np.isnan(X_tr).any() or np.isnan(X_te).any(), "   memoria train: %.0f MB" % (X_tr.nbytes / 1e6))

# --- controllo del rango di X^T X ---
d = X_tr.shape[1]
G = np.zeros((d, d))
for s in range(0, X_tr.shape[0], 50_000):
    B = X_tr[s:s + 50_000].astype(np.float64)
    G += B.T @ B
dg = np.sqrt(np.diag(G))
autoval, autovet = np.linalg.eigh(G / np.outer(dg, dg))

nulli = np.where(autoval < 1e-8 * autoval[-1])[0]
print(f"\nrango numerico: {d - len(nulli)} su {d} colonne")
print(f"numero di condizionamento: {autoval[-1] / autoval[0]:.2e}" if autoval[0] > 0 else "numero di condizionamento: infinito")
if len(nulli) == 0:
    print("--> RANGO PIENO: nessuna collinearità esatta, X^T X è invertibile")
for k in nulli:
    v = autovet[:, k]
    top = [j for j in np.argsort(-np.abs(v))[:6] if abs(v[j]) > 0.05]
    print(f"   autovalore {autoval[k]:.1e}: " + ", ".join(f"{nomi[j]} ({v[j]:+.2f})" for j in top))

print("\ni 10 autovalori più piccoli:", ", ".join(f"{a:.1e}" for a in autoval[:10]))
print("colonne coinvolte nel più piccolo:",
      ", ".join(f"{nomi[j]} ({autovet[j, 0]:+.2f})" for j in np.argsort(-np.abs(autovet[:, 0]))[:6]))